In [ ]:
#| default_exp mcp

## MCP server

Expose nbskill notebook operations as native MCP tools. This is the preferred integration for careful single-notebook reads and edits because multiline notebook cells travel as structured tool arguments rather than shell-quoted strings.

The server now favors verifiable context: batch edits include read-back hashes, symbol graph calls include complete structured usage data, and generated-file warnings are reserved for changes that touch exported notebook code.

The command-line functions are useful on their own, but coding agents work best when the same operations are available as structured tools. This notebook exposes the project through a FastMCP server while keeping the server layer thin and predictable.

### Production contract

The MCP server exposes the production core through stable structured tools. Tool schemas must hide CLI-only flags, responses must include concise text plus structured content, diagnostics must be scoped to touched notebooks when possible, edit tools must report feedback without raw notebook JSON, and experimental tools must be clearly marked or omitted from default production use.


The MCP server should stay boring on purpose. Each tool accepts structured arguments, captures printed output, uses notebook locks where file operations can collide, and delegates the actual work to the same functions tested elsewhere.

```python
mcp = create_mcp()
# MCP clients see tools such as nb_overview, nb_chapter, nb_cell, write_nb, update_cell, exec_nb, and diff_nb.
```

In [ ]:
from contextlib import redirect_stdout
from io import StringIO
import nbskill.mcp as _mcp_mod
from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import write_nb as _write_raw_nb
from nbskill.mcp import capture_call as _example_capture_call
from nbskill.mcp import create_mcp as _example_create_mcp
from nbskill.read import nb_overview as _example_nb_overview
from nbskill.write import write_nb as _example_write_nb
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook

In [ ]:
def demo_tool():
    print("captured output")

print(_example_capture_call(demo_tool))
print(type(_example_create_mcp()).__name__)

captured output
FastMCP


In [ ]:
#| export
import hashlib,json,os
import re
import shutil
import subprocess
import sys
import threading
import time
from contextlib import redirect_stdout, redirect_stderr
from importlib.metadata import PackageNotFoundError, version
from io import StringIO
from pathlib import Path


In [ ]:
#| export
from fastcore.nbio import read_nb as _read_raw_nb
from fastcore.script import Param, call_parse, _in_call_parse
from fastmcp import FastMCP
from fastmcp.tools import ToolResult
from mcp.types import TextContent


In [ ]:
#| export
from nbskill.convert import new_nbdev_notebook
from nbskill.convert import py2nb
from nbskill.convert import py2nbdev
from nbskill.edit_interactive import execute_plan
from nbskill.edit_interactive import execute_project_plan
from nbskill.edit_interactive import plan_result_text
from nbskill.execute import exec_nb
from nbskill.workbench import agent_workbench_result
from nbskill.foundation import empty_failure_map, failure_map_path, load_failure_map
from nbskill.foundation import cell_source, exported_py_path, path_candidates


In [ ]:
#| export
from nbskill.graph import notebook_order_problems
from nbskill.graph import private_symbol_report
from nbskill.graph import symbol_graph_data, symbol_graph_public_data
from nbskill.graph import symbol_graph
from nbskill.knowledge import add_behaviour_steering
from nbskill.knowledge import get_knowledge
from nbskill.knowledge import store_knowledge
from nbskill.parallel import notebook_locks
from nbskill.read import nb_cell
from nbskill.read import nb_chapter
from nbskill.read import nb_overview
from nbskill.read import show_doc


In [ ]:
#| export
from nbskill.review import reset_global_usage_summary
from nbskill.review import notebook_size_problems
from nbskill.review import notebook_validation_problems
from nbskill.review import run_style_check
from nbskill.review import style_check
from nbskill.review import style_report
from nbskill.review import diff_nb
from nbskill.write import apply_notebook_edit
from nbskill.write import insert_notebook_cells
from nbskill.write import replace_notebook_cell
from nbskill.write import replace_notebook_range
from nbskill.write import should_run_cell_feedback
from nbskill.write import source_lines_cells


### Capturing command output

The MCP tools should return text, not leak stdout and stderr into the server process. These helpers capture each underlying function call and convert its visible result into one response string.

In [ ]:
#| export
def as_text(value):
    return "" if value is None else str(value)


In [ ]:
#| export
_CAPTURE_LOCK = threading.RLock()


In [ ]:
#| export
_REDACT_KEYS = {"new", "new_lines", "source_lines", "replacement_lines", "cells", "edits", "operations", "source", "old_str", "new_str"}


In [ ]:
#| export
_REMOVED_SCRIPT_NAMES = (
    "nbskill-mcp", "read-nb", "write-nb", "update-cell", "batch-edit-nb",
    "show-doc", "exec-nb", "diff-nb", "style-check", "symbol-graph", "private-symbol-report",
)


In [ ]:
#| export
_GENERATED_RE = re.compile(r"^# AUTOGENERATED! DO NOT EDIT! File to edit: (.+)$")


In [ ]:
#| export
def _package_version(name="nbskill"):
    try: return version(name)
    except PackageNotFoundError: return "unknown"


In [ ]:
#| export
def capture_call(func, **kwargs):
    out, err = StringIO(), StringIO()
    with _CAPTURE_LOCK:
        original_stdout, original_stderr = sys.stdout, sys.stderr
        try:
            try:
                with redirect_stdout(out), redirect_stderr(err):
                    result = func(**kwargs)
            except SystemExit as exc:
                chunks = []
                if out.getvalue(): chunks.append(out.getvalue().rstrip())
                if err.getvalue(): chunks.append(err.getvalue().rstrip())
                chunks.append(f"SystemExit: {exc.code}")
                raise RuntimeError(chr(10).join(chunk for chunk in chunks if chunk)) from exc
        finally:
            sys.stdout, sys.stderr = original_stdout, original_stderr
    chunks = []
    if out.getvalue(): chunks.append(out.getvalue().rstrip())
    if err.getvalue(): chunks.append(err.getvalue().rstrip())
    if result is not None and not chunks: chunks.append(as_text(result))
    return chr(10).join(chunk for chunk in chunks if chunk)


In [ ]:
#| export
def capture_notebook_call(func, *paths, **kwargs):
    "Capture a call while holding per-notebook locks for `paths`."
    with notebook_locks(*paths):
        return capture_call(func, **kwargs)


In [ ]:
#| export
def _mcp_find_cell(path, cell_id):
    nb = _read_raw_nb(path)
    for cell in nb.cells:
        if getattr(cell, "id", None) == cell_id: return cell
    raise ValueError(f"Cell id {cell_id!r} was not found in {path}")


In [ ]:
#| export
def _mcp_source_hash(source):
    return hashlib.sha256(str(source).encode("utf-8")).hexdigest()[:12]


In [ ]:
#| export
def _mcp_cell_source_hash(path, cell_id):
    return _mcp_source_hash(cell_source(_mcp_find_cell(path, cell_id)))


In [ ]:
#| export
def _mcp_expected_hash_warning(path, cell_id, expected_hash):
    if not expected_hash: return None
    actual = _mcp_cell_source_hash(path, cell_id)
    if actual == expected_hash: return None
    return _warning(
        "expected_hash_mismatch",
        f"Skipped edit for {path} id={cell_id}: expected hash {expected_hash}, found {actual}.",
        "Refresh nb_cell context and retry with the current source.",
        path=str(path), cell_id=cell_id, expected_hash=expected_hash, actual_hash=actual,
    )


In [ ]:
#| export
def _capture_exec_nb_cli_call(arguments):
    call_args = {key: value for key, value in arguments.items() if key != "detail"}
    script = "; ".join([
        "import json, sys",
        "from nbskill.execute import exec_nb",
        "exec_nb(**json.loads(sys.argv[1]))",
    ])
    lock_paths = [call_args.get("path")]
    if call_args.get("dest"): lock_paths.append(call_args["dest"])
    with notebook_locks(*lock_paths):
        proc = subprocess.run(
            [sys.executable, "-c", script, json.dumps(call_args)],
            text=True, capture_output=True, cwd=str(Path.cwd()),
        )
    chunks = [item.rstrip() for item in (proc.stdout, proc.stderr) if item]
    output = chr(10).join(chunks)
    if proc.returncode != 0:
        raise RuntimeError(output or f"exec_nb subprocess failed with exit code {proc.returncode}")
    return output


In [ ]:
#| export
def _mcp_cell_index(path, cell_id):
    nb = _read_raw_nb(path)
    for index, cell in enumerate(nb.cells):
        if getattr(cell, "id", None) == cell_id: return index
    raise ValueError(f"Cell id {cell_id!r} was not found in {path}")


In [ ]:
#| export
def _mcp_cell_ids_from_index(path, start, count):
    if start is None or count <= 0: return []
    nb = _read_raw_nb(path)
    return [
        getattr(cell, "id", None) for cell in nb.cells[start:start + count]
        if getattr(cell, "id", None)
    ]


In [ ]:
#| export
def _mcp_structured_cell_count(cells, default_cell_type="code"):
    return sum(len(source_lines_cells(cell, default_cell_type)) for cell in (cells or []))


In [ ]:
#| export
def _mcp_feedback_location(path, edit, default_cell_type="code"):
    edit_path = str(edit.get("path") or path)
    op = edit.get("op")
    if op == "replace_cell":
        cell_id = edit.get("cell_id")
        return edit_path, _mcp_cell_index(edit_path, cell_id), _mcp_structured_cell_count([edit], default_cell_type)
    if op == "replace_range":
        return edit_path, _mcp_cell_index(edit_path, edit.get("cell_id")), 1
    if op in {"insert_before", "insert_after"}:
        anchor_id = edit.get("anchor_id") or edit.get("cell_id")
        anchor_index = _mcp_cell_index(edit_path, anchor_id)
        start = anchor_index if op == "insert_before" else anchor_index + 1
        return edit_path, start, _mcp_structured_cell_count(edit.get("cells") or [edit], default_cell_type)
    return edit_path, None, 0


In [ ]:
#| export
def _mcp_feedback_output(path, cell_ids, auto_feedback=True, feedback_timeout=10, feedback_safe=True):
    if not auto_feedback: return ""
    nb = _read_raw_nb(path)
    cells_by_id = {getattr(cell, "id", None): cell for cell in nb.cells}
    target_id = None
    for cell_id in cell_ids:
        cell = cells_by_id.get(cell_id)
        if cell is not None and should_run_cell_feedback(cell): target_id = cell_id
    if target_id is None: return ""
    output = _capture_exec_nb_cli_call(dict(
        path=str(path), dest=None, exc_stop=False, up2id=target_id, chapter=None,
        timeout=feedback_timeout, show_output=True, verbose=False, safe=feedback_safe,
        allow=None, ok_dests=None, cache_httpx=False, cache_dir=None,
        cache_domains=None, allow_new=True, check_only=True,
    ))
    return f"Auto feedback (up to id={target_id}):\n{output}".rstrip()


In [ ]:
#| export
def _append_mcp_feedback(message, path, cell_ids, auto_feedback=True, feedback_timeout=10, feedback_safe=True):
    feedback = _mcp_feedback_output(path, cell_ids, auto_feedback, feedback_timeout, feedback_safe)
    return "\n\n".join(chunk for chunk in [message.rstrip(), feedback] if chunk)


In [ ]:
#| export
def _json_preview(value, limit=1200):
    text = json.dumps(value, indent=2, sort_keys=True, default=str)
    if len(text) <= limit: return text
    return f"{text[:limit].rstrip()}\n... truncated ..."


In [ ]:
#| export
def _text_preview(value, limit=12000):
    text = as_text(value)
    if limit is None or len(text) <= limit:
        return {"text": text, "truncated": False, "chars": len(text), "omitted_chars": 0}
    omitted = len(text) - limit
    return {
        "text": f"{text[:limit].rstrip()}\n... truncated {omitted} chars ...",
        "truncated": True,
        "chars": len(text),
        "omitted_chars": omitted,
    }


In [ ]:
#| export
def _redact_value(key, value, limit=160):
    if value is None: return None
    text = as_text(value)
    if key in _REDACT_KEYS and len(text) > limit:
        return f"<{len(text)} chars redacted; use detail='debug' to inspect>"
    if len(text) > limit * 3:
        return f"{text[:limit].rstrip()}... <{len(text) - limit} more chars>"
    return value


In [ ]:
#| export
def _redact_arguments(arguments):
    return {key: _redact_value(key, value) for key, value in (arguments or {}).items()}


In [ ]:
#| export
def _warning(code, message, next_action=None, **extra):
    item = {"code": code, "message": message}
    if next_action: item["next_action"] = next_action
    item.update({key: value for key, value in extra.items() if value is not None})
    return item


In [ ]:
#| export
def _path_without_cwd_prefix(path):
    return path_candidates(path)[-1]


In [ ]:
#| export
def _git_base(path="."):
    base = _path_without_cwd_prefix(path)
    if (base.exists() and not base.is_dir()) or (not base.exists() and base.suffix):
        return base.parent
    return base


In [ ]:
#| export
def _git_root(path="."):
    for base in dict.fromkeys([_git_base(path), Path(".")]):
        proc = subprocess.run(["git", "-C", str(base), "rev-parse", "--show-toplevel"], text=True, capture_output=True)
        if proc.returncode == 0 and proc.stdout.strip(): return Path(proc.stdout.strip())
    return None


In [ ]:
#| export
def _rooted_path(path, root):
    raw = Path(str(path)).expanduser()
    pth = _path_without_cwd_prefix(path)
    candidates = [pth]
    if root is not None and not raw.is_absolute():
        candidates = [Path(root) / raw, Path(root) / pth, *candidates]
    for candidate in candidates:
        try:
            if candidate.exists(): return candidate.resolve()
        except OSError:
            continue
    return pth.resolve()


In [ ]:
#| export
def _rel_to_root(path, root):
    try: return _rooted_path(path, root).relative_to(Path(root).resolve()).as_posix()
    except (OSError, ValueError): return str(path)


In [ ]:
#| export
def _git_changed_paths(root):
    proc = subprocess.run(["git", "-C", str(root), "status", "--porcelain"], text=True, capture_output=True)
    if proc.returncode != 0: return set()
    paths = set()
    for line in proc.stdout.splitlines():
        raw = line[3:].strip()
        if " -> " in raw: raw = raw.split(" -> ", 1)[1]
        if raw: paths.add(raw)
    return paths


In [ ]:
#| export
def _generated_owner(path):
    path = Path(path)
    if path.suffix != ".py" or not path.exists(): return None
    try:
        for line in path.read_text(encoding="utf-8", errors="ignore").splitlines()[:3]:
            match = _GENERATED_RE.match(line.strip())
            if match: return (path.parent / match.group(1).rstrip(".")).resolve()
    except OSError:
        return None
    return None


In [ ]:
#| export
def _generated_files(root):
    skip = {".git", ".venv", "__pycache__", ".mypy_cache", ".pytest_cache"}
    items = []
    for path in Path(root).rglob("*.py"):
        if any(part in skip for part in path.parts): continue
        owner = _generated_owner(path)
        if owner is not None: items.append((path, owner))
    return items


In [ ]:
#| export
def _owner_output(path):
    owner = _generated_owner(path)
    if owner is None: return f"No generated-notebook owner found for {path}"
    return f"Generated file owner: {path} -> {owner}"


In [ ]:
#| export
def _failure_data():
    path = failure_map_path()
    try: return load_failure_map(path) if path.exists() else empty_failure_map()
    except OSError: return empty_failure_map()


In [ ]:
#| export
def _resolve_diagnostic_scope(path, root, scope_path=None):
    raw = path if scope_path in (None, "") else scope_path
    if raw in (None, "", "."): return Path(root)
    scoped = Path(raw).expanduser()
    if not scoped.is_absolute(): scoped = Path(root) / scoped
    return scoped


In [ ]:
#| export
def _is_project_scope(path, root):
    try: return Path(path).resolve() == Path(root).resolve()
    except OSError: return False


In [ ]:
#| export
def _scope_notebook_paths(path):
    path = Path(path)
    if path.suffix == ".ipynb": return [path]
    if path.is_dir():
        return sorted(
            item for item in path.rglob("*.ipynb")
            if ".ipynb_checkpoints" not in item.parts
        )
    return []


In [ ]:
#| export
def _generated_pairs_for_scope(root, path):
    path = Path(path)
    if _is_project_scope(path, root): return _generated_files(root)
    if path.suffix == ".py":
        owner = _generated_owner(path)
        return [(path, owner)] if owner is not None else []
    pairs = []
    for nb_path in _scope_notebook_paths(path):
        try: py_path = exported_py_path(nb_path)
        except (FileNotFoundError, OSError): py_path = None
        if py_path is not None and Path(py_path).exists(): pairs.append((Path(py_path), nb_path.resolve()))
    return pairs


In [ ]:
#| export
def _cell_source_text(cell):
    if isinstance(cell, dict):
        source = cell.get("source", "")
        return "".join(source) if isinstance(source, list) else str(source)
    return cell_source(cell)


In [ ]:
#| export
def _cell_type_text(cell):
    return cell.get("cell_type", "") if isinstance(cell, dict) else getattr(cell, "cell_type", "")


In [ ]:
#| export
def _cell_export_relevant(cell):
    if _cell_type_text(cell) != "code": return False
    return any(re.match(r"^\s*#\|\s*(export|default_exp)\b", line) for line in _cell_source_text(cell).splitlines())


In [ ]:
#| export
def _notebook_export_relevant_change(owner, root):
    owner = Path(owner)
    root = Path(root)
    owner_rel = _rel_to_root(owner, root)
    proc = subprocess.run(["git", "-C", str(root), "show", f"HEAD:{owner_rel}"], text=True, capture_output=True)
    if proc.returncode != 0: return True
    try:
        old_nb = json.loads(proc.stdout)
        new_nb = json.loads(owner.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return True
    old_cells = {cell.get("id"): cell for cell in old_nb.get("cells", []) if cell.get("id")}
    new_cells = {cell.get("id"): cell for cell in new_nb.get("cells", []) if cell.get("id")}
    for cell_id in set(old_cells) | set(new_cells):
        old_cell, new_cell = old_cells.get(cell_id), new_cells.get(cell_id)
        if old_cell is None or new_cell is None:
            changed = True
        else:
            changed = _cell_type_text(old_cell) != _cell_type_text(new_cell) or _cell_source_text(old_cell) != _cell_source_text(new_cell)
        if changed and any(_cell_export_relevant(cell) for cell in (old_cell, new_cell) if cell is not None):
            return True
    return False


In [ ]:
#| export
def _style_problem_warnings(path, root):
    warnings = []
    for problem in notebook_size_problems(path):
        code = problem.get("code")
        if code == "large-cell":
            cell = f" cell id={problem['cell_id']}" if problem.get("cell_id") else ""
            warnings.append(_warning(
                "large_cell",
                f"Notebook {_rel_to_root(problem.get('path'), root)}{cell} is large: {problem.get('detail')}.",
                "Split the cell so it contains one idea before continuing.",
                path=problem.get("path"), cell_id=problem.get("cell_id"), problem=problem,
            ))
        elif code == "large-generated-py":
            generated = problem.get("exported_py_path")
            warnings.append(_warning(
                "large_generated_py",
                f"Generated file {_rel_to_root(generated, root)} is getting large: {problem.get('detail')}.",
                "Use the notebook split tool to split the source notebook/module.",
                path=problem.get("path"), generated=generated, problem=problem,
            ))
    return warnings


In [ ]:
#| export
def _warning_scope_root_rel(path, root):
    if path in (None, ""): return ""
    try:
        return Path(path).expanduser().resolve().relative_to(Path(root).resolve()).as_posix()
    except (OSError, ValueError):
        return str(path).strip()


In [ ]:
#| export
def _warning_scope_matches(value, root, scope):
    if scope is None: return True
    rel = _warning_scope_root_rel(value, root)
    return any(rel == item or rel.startswith(f"{item.rstrip('/')}/") for item in scope)


In [ ]:
#| export
def _warning_related_paths(item):
    for key in ("path", "generated", "owner"):
        if item.get(key): yield item[key]
    problem = item.get("problem") or {}
    for key in ("path", "exported_py_path"):
        if problem.get(key): yield problem[key]


In [ ]:
#| export
def _warning_cell_id(item):
    problem = item.get("problem") or {}
    return item.get("cell_id") or problem.get("cell_id")


In [ ]:
#| export
_NOTEBOOK_LEVEL_CONTEXT_WARNING_CODES = {
    "generated_without_notebook", "notebook_export_missing", "exported_py_hash_mismatch",
}


In [ ]:
#| export
def _filter_warnings_for_scope(warnings, root, scope_path=None, scope_cell_ids=None):
    if scope_path in (None, "", ".") and scope_cell_ids is None: return warnings
    scope = None if scope_path in (None, "", ".") else {_warning_scope_root_rel(scope_path, root).rstrip("/")}
    cell_filter_active = scope_cell_ids is not None
    cell_ids = set(scope_cell_ids or [])
    filtered = []
    for item in warnings:
        if scope is not None and not any(_warning_scope_matches(path, root, scope) for path in _warning_related_paths(item)):
            continue
        cell_id = _warning_cell_id(item)
        if cell_filter_active and cell_id and cell_id not in cell_ids: continue
        if cell_filter_active and not cell_id and item.get("code") not in _NOTEBOOK_LEVEL_CONTEXT_WARNING_CODES: continue
        filtered.append(item)
    return filtered


In [ ]:
#| export
def _doctor_warnings(path=".", scope_path=None, scope_cell_ids=None):
    root = _git_root(path) or _git_base(path).resolve()
    diagnostic_path = _resolve_diagnostic_scope(path, root, scope_path)
    project_scope = _is_project_scope(diagnostic_path, root)
    changed = _git_changed_paths(root) if (root / ".git").exists() else set()
    warnings = []
    for py_path, owner in _generated_pairs_for_scope(root, diagnostic_path):
        py_rel = _rel_to_root(py_path, root)
        owner_rel = _rel_to_root(owner, root)
        if py_rel in changed and owner_rel not in changed:
            warnings.append(_warning(
                "generated_without_notebook",
                f"Generated file {py_rel} changed without its source notebook {owner_rel}.",
                "Move the edit into the notebook and export, or verify the generated edit is intentional.",
                path=py_rel, owner=owner_rel,
            ))
        if owner_rel in changed and py_rel not in changed:
            if _notebook_export_relevant_change(owner, root):
                warnings.append(_warning(
                    "notebook_export_missing",
                    f"Notebook {owner_rel} changed but generated file {py_rel} is unchanged.",
                    "Run export or use an nbskill write tool before shipping.",
                    path=owner_rel, generated=py_rel,
                ))
    for problem in notebook_validation_problems(diagnostic_path):
        if problem.get("code") != "exported-py-hash-mismatch": continue
        warnings.append(_warning(
            "exported_py_hash_mismatch",
            f"Notebook {problem['path']} metadata does not match current generated file {problem.get('exported_py_path')}.",
            "Run nbskill_validate or export the notebook through nbskill write tools.",
            path=problem.get("path"), generated=problem.get("exported_py_path"),
        ))
    warnings.extend(_style_problem_warnings(diagnostic_path, root))
    if project_scope:
        warnings.extend(_doc_script_warnings(root))
        failures = _failure_data().get("events", [])[-10:]
        recent_failures = [event for event in failures if event.get("kind") == "failure"]
        if recent_failures:
            last = recent_failures[-1]
            warnings.append(_warning(
                "recent_tool_failures",
                f"Recent nbskill failure: {last.get('tool')} {last.get('summary') or last.get('error')}",
                "Run doctor(detail='debug') or style_check(delete_after_output=True) after resolving it.",
                tool=last.get("tool"),
            ))
    if scope_path is None: scope_path = path
    return _filter_warnings_for_scope(warnings, root, scope_path, scope_cell_ids)


In [ ]:
#| export
def _doc_script_warnings(root):
    docs = [Path(root) / "README.md", Path(root) / "nbskill" / "SKILL.md"]
    docs += list((Path(root) / "nbskill" / "references").glob("*.md")) if (Path(root) / "nbskill" / "references").exists() else []
    warnings = []
    for doc in docs:
        if not doc.exists(): continue
        try: text = doc.read_text(encoding="utf-8", errors="ignore")
        except OSError: continue
        found = sorted(name for name in _REMOVED_SCRIPT_NAMES if name in text)
        if found:
            warnings.append(_warning(
                "removed_script_name",
                f"{doc.relative_to(root)} references removed CLI names: {', '.join(found)}.",
                "Replace hyphenated command names with underscore script names.",
                path=str(doc), names=found,
            ))
    return warnings


In [ ]:
#| export
def _first_match(pattern, text, flags=0):
    match = re.search(pattern, text or "", flags)
    return match.group(1) if match else None


In [ ]:
#| export
def _line_cell_ids(text):
    return set(re.findall(r"^Cell id=([^\s:]+)", text or "", re.MULTILINE))


In [ ]:
#| export
def _response_scope_cell_ids(tool, arguments, preview):
    text = preview.get("text", "")
    if tool == "nb_overview": return None
    if tool == "nb_chapter": return _line_cell_ids(text) or None
    if tool == "nb_cell":
        if arguments.get("id"): return {str(arguments["id"])}
        cell_id = _first_match(r"^Cell id=([^\s:]+)", text, re.MULTILINE)
        return {cell_id} if cell_id else None
    if tool == "show_doc":
        cell_id = _first_match(r"^Location:.*?Cell id=([^\s:]+)", text, re.MULTILINE)
        return {cell_id} if cell_id else None
    if tool == "diff_nb":
        ids = set(re.findall(r"--- code cell ([^\s]+) ---", text))
        return ids if ids else set()
    if tool in {"edit_cell", "edit_cell_range"}:
        cell_id = arguments.get("cell_id") or _first_match(r"Updated .* id=([^\s]+)", text)
        return {str(cell_id)} if cell_id else None
    if tool == "insert_cells":
        anchor_id = arguments.get("anchor_id")
        return {str(anchor_id)} if anchor_id else None
    if tool == "apply_notebook_edits":
        ids = {str(item.get("cell_id") or item.get("anchor_id")) for item in arguments.get("edits", []) if isinstance(item, dict) and (item.get("cell_id") or item.get("anchor_id"))}
        return ids or None
    if tool == "exec_nb":
        up2id = arguments.get("up2id")
        if isinstance(up2id, str) and up2id and not up2id.isdigit(): return {up2id}
        return _line_cell_ids(text) or None
    for key in ("id", "cell_id", "any_cell_id"):
        value = arguments.get(key)
        if value: return {str(value)}
    return None


In [ ]:
#| export
def _unique_strings(items):
    seen, result = set(), []
    for item in items:
        if item in seen: continue
        seen.add(item)
        result.append(item)
    return result


In [ ]:
#| export
def _notebook_paths_from_text(text):
    return _unique_strings(re.findall(r"(?<![\w.-])(?:[~./A-Za-z0-9_-]+\.ipynb)", text or ""))


In [ ]:
#| export
def _edit_scope_paths(arguments):
    default = arguments.get("path")
    edits = arguments.get("edits") or []
    paths = [default] if default else []
    paths += [item.get("path") for item in edits if isinstance(item, dict) and item.get("path")]
    return _unique_strings(str(item) for item in paths if item)


In [ ]:
#| export
def _response_scope_paths(tool, arguments, preview):
    text_paths = _notebook_paths_from_text(preview.get("text", ""))
    if tool == "apply_notebook_edits":
        return _unique_strings([*_edit_scope_paths(arguments), *text_paths])
    if tool in {"edit_cell", "edit_cell_range", "insert_cells"} and text_paths:
        return text_paths
    path = arguments.get("path") or arguments.get("nb_path")
    return [str(path)] if path else []


In [ ]:
#| export
def _dedupe_warnings(warnings):
    seen, result = set(), []
    for item in warnings:
        key = json.dumps(item, sort_keys=True, default=str)
        if key in seen: continue
        seen.add(key)
        result.append(item)
    return result


In [ ]:
#| export
def _response_warnings(tool, arguments, preview):
    warnings = []
    if preview.get("truncated"):
        warnings.append(_warning(
            "output_truncated",
            f"{tool} output was truncated by {preview['omitted_chars']} chars.",
            "Repeat with a narrower query or detail='debug' if you need full context.",
        ))
    context_tools = {"nb_overview", "nb_chapter", "nb_cell", "show_doc", "diff_nb"}
    scoped_project_tools = {"edit_cell", "edit_cell_range", "insert_cells", "apply_notebook_edits", "exec_nb"}
    if tool in context_tools:
        path = arguments.get("path") or arguments.get("nb_path") or "."
        cell_ids = _response_scope_cell_ids(tool, arguments, preview)
        warnings.extend(_doctor_warnings(path, scope_path=path, scope_cell_ids=cell_ids)[:3])
    elif tool in scoped_project_tools:
        cell_ids = _response_scope_cell_ids(tool, arguments, preview)
        for path in _response_scope_paths(tool, arguments, preview):
            warnings.extend(_doctor_warnings(path, scope_path=path, scope_cell_ids=cell_ids))
    return _dedupe_warnings(warnings)[:3]


In [ ]:
#| export
def _brief_call(tool, arguments, preview):
    lines = [f"{tool} completed"]
    for key in ("path", "nb_path", "cell_id", "id", "chapter", "name", "any_cell_id", "symbol"):
        if arguments.get(key) not in (None, ""):
            lines.append(f"{key}={arguments[key]}")
    if arguments.get("dry_run") is True: lines.append("dry_run=True")
    if preview.get("truncated"): lines.append(f"output_truncated=True omitted_chars={preview['omitted_chars']}")
    return lines


In [ ]:
#| export
def mcp_tool_result(tool, arguments, full_output, max_output_chars=12000, detail="summary", warnings=None, hints=None, **structured):
    "Return concise visible MCP text plus structured data for clients that inspect it."
    detail = detail or "summary"
    preview = _text_preview(full_output or "", limit=max_output_chars)
    warnings = [*(warnings or []), *_response_warnings(tool, arguments or {}, preview)]
    hints = list(hints or [])
    lines = _brief_call(tool, arguments or {}, preview)
    if preview["text"]:
        lines += ["", "Result:", preview["text"]]
    if warnings:
        lines += ["", "Warnings:"]
        lines.extend(f"- {item['message']}" + (f" Next: {item['next_action']}" if item.get("next_action") else "") for item in warnings)
    if detail == "debug" and hints:
        lines += ["", "Hints:"]
        lines.extend(f"- {hint}" for hint in hints)
    summary = "\n".join(lines)
    data = {
        "summary": summary,
        "call": {"tool": tool, "arguments": _redact_arguments(arguments or {})},
        "full_output": preview["text"],
        "output_truncated": preview["truncated"],
        "output_chars": preview["chars"],
        "omitted_chars": preview["omitted_chars"],
        "warnings": warnings,
        "hints": hints if detail == "debug" else [],
    }
    if detail == "debug":
        data["debug"] = {"arguments": arguments or {}, "raw_output": as_text(full_output or "")}
    data.update(structured)
    return ToolResult(content=[TextContent(type="text", text=summary)], structured_content=data)


In [ ]:
#| export
def _status_data():
    scripts = [
        "nb_overview", "nb_chapter", "nb_cell", "write_nb", "update_cell",
        "batch_edit_nb", "show_doc", "exec_nb", "diff_nb", "style_check",
        "install_nbskill", "symbol_graph", "private_symbol_report", "agent_workbench",
        "new_nbdev_notebook", "add_behaviour_steering", "store_knowledge", "get_knowledge",
        "nbskill_mcp",
    ]
    return {
        "version": _package_version(),
        "cwd": str(Path.cwd()),
        "python": sys.executable,
        "mcp_command": "nbskill_mcp",
        "mcp_command_path": shutil.which("nbskill_mcp"),
        "cli_tools": {name: shutil.which(name) for name in scripts},
        "reconnect_hint": "Restart or reconnect the MCP client after reinstalling nbskill or changing tool signatures.",
        "install_commands": [
            "uv tool install --editable . --force",
            "codex mcp add nbskill -- nbskill_mcp",
            "claude mcp add nbskill -- nbskill_mcp",
        ],
    }


In [ ]:
#| export
def _format_status(data):
    lines = [
        "nbskill status",
        f"version={data['version']}",
        f"cwd={data['cwd']}",
        f"python={data['python']}",
        f"mcp_command={data['mcp_command']}",
        f"mcp_command_path={data['mcp_command_path'] or '(not on PATH)'}",
        "cli_tools:",
    ]
    lines.extend(f"- {name}: {path or '(not on PATH)'}" for name, path in data["cli_tools"].items())
    lines.append(f"reconnect_hint={data['reconnect_hint']}")
    lines.append("install_commands:")
    lines.extend(f"- {cmd}" for cmd in data["install_commands"])
    return "\n".join(lines)


In [ ]:
#| export
def _doctor_report(
    path=".",
    detail="summary",
    fix=False,
    reset=False,
    capabilities="",
    scopes="error,warning",
    skip_folder_re=None,
    skip_path=None,
    max_output_chars=12000,
    max_diagnostics=200,
):
    root = _git_root(path) or _git_base(path).resolve()
    status = _status_data()
    selected = _doctor_scope_set(scopes)
    errors = _doctor_error_items(path, status) if "error" in selected else []
    warnings, private_text = _doctor_warning_items(path) if "warning" in selected else ([], "")
    style = (
        _doctor_style_report(path, skip_folder_re=skip_folder_re, skip_path=skip_path, max_output_chars=max_output_chars, max_diagnostics=max_diagnostics)
        if "style" in selected else None
    )
    failure_map = _failure_data()
    if reset: reset_global_usage_summary()
    hints = [
        "Use scopes='error,warning,style' or scopes='all' for the full doctor report.",
        "Use scopes='style' to include chkstyle output; chkstyle is omitted from error/warning scopes.",
        "Use nb_overview/nb_chapter/nb_cell/show_doc, then edit_cell, edit_cell_range, insert_cells, or apply_notebook_edits.",
    ]
    changed = sorted(_git_changed_paths(root)) if (root / ".git").exists() else []
    generated = [
        {"path": _rel_to_root(py, root), "owner": _rel_to_root(owner, root)}
        for py, owner in _generated_files(root)
    ]
    issue_count = len(errors) + len(warnings)
    style_count = (style or {}).get("summary", {}).get("diagnostic_count", 0)
    summary = f"nbskill doctor: {len(errors)} error(s), {len(warnings)} warning(s)"
    if "style" in selected: summary += f", {style_count} style diagnostic(s)"
    if not issue_count and "style" not in selected: summary = "nbskill doctor: no actionable errors or warnings"
    text_lines = [summary]
    if errors:
        text_lines.append("\nErrors:")
        text_lines.extend(f"- {item['message']}" + (f" Next: {item['next_action']}" if item.get("next_action") else "") for item in errors)
    if warnings:
        text_lines.append("\nWarnings:")
        text_lines.extend(f"- {item['message']}" + (f" Next: {item['next_action']}" if item.get("next_action") else "") for item in warnings)
    if style is not None and style.get("text", "").strip():
        text_lines.append("\nStyle:")
        text_lines.append(style["text"].strip())
    report = {
        "path": str(path),
        "root": str(root),
        "scopes": sorted(selected),
        "status": status,
        "errors": errors,
        "warnings": warnings,
        "issues": [*errors, *warnings],
        "style": style,
        "private_symbol_report": private_text if "warning" in selected else "",
        "hints": hints,
        "capabilities": [item for item in _MCP_CAPABILITIES.split(",") if item],
        "changed_paths": changed if detail == "debug" else changed[:20],
        "generated_owners": generated if detail == "debug" else generated[:20],
        "recent_events": failure_map.get("events", [])[-20:] if detail == "debug" else [],
        "counts": failure_map.get("counts", {}),
        "reset": bool(reset),
        "fix": {"requested": bool(fix), "applied": []},
        "text": "\n".join(text_lines),
    }
    return report


In [ ]:
#| export
@call_parse
def nbskill_status(json_output: bool = False):  # Print JSON instead of text
    "Report nbskill version, MCP command setup, canonical CLI tools, and reconnect hints."
    data = _status_data()
    print(json.dumps(data, indent=2, sort_keys=True) if json_output else _format_status(data))
    return data if not _in_call_parse else None


In [ ]:
#| export
_DOCTOR_SCOPES = {"error", "warning", "style"}


In [ ]:
#| export
def _doctor_scope_set(scopes="error,warning"):
    "Normalize comma/space-separated doctor scopes."
    if scopes is None: return {"error", "warning"}
    raw = str(scopes).replace(",", " ").split()
    selected = set(raw) or {"error", "warning"}
    if "all" in selected: return set(_DOCTOR_SCOPES)
    unknown = selected - _DOCTOR_SCOPES
    if unknown: raise ValueError(f"Unknown doctor scope(s): {', '.join(sorted(unknown))}")
    return selected


In [ ]:
#| export
def _problem_message(problem):
    parts = [str(problem.get("path") or "")]
    if problem.get("cell_id"): parts.append(f"id={problem['cell_id']}")
    if problem.get("line"): parts.append(f"line={problem['line']}")
    if problem.get("symbol"): parts.append(f"symbol={problem['symbol']!r}")
    if problem.get("code"): parts.append(f"code={problem['code']}")
    if problem.get("detail"): parts.append(str(problem["detail"]))
    return " ".join(part for part in parts if part)


In [ ]:
#| export
def _doctor_validation_errors(path):
    errors = []
    for problem in notebook_validation_problems(path):
        errors.append(_warning(
            problem.get("code", "notebook-validation"),
            _problem_message(problem),
            "Fix notebook metadata/source ordering before relying on notebook edits.",
            severity="error", scope="error", problem=problem,
        ))
    return errors


In [ ]:
#| export
def _doctor_order_errors(path):
    errors = []
    try:
        problems = notebook_order_problems(path)
    except FileNotFoundError as exc:
        return [_warning(
            "notebook_order_missing_file",
            f"Notebook order scan found a missing notebook: {exc.filename or exc}",
            "Remove stale notebook references or recreate the missing notebook before rerunning doctor.",
            severity="error", scope="error", path=exc.filename,
        )]
    for problem in problems:
        errors.append(_warning(
            problem.get("code", "notebook-order"),
            _problem_message(problem),
            "Move definitions/imports before use, or add the missing import.",
            severity="error", scope="error", problem=problem,
        ))
    return errors


In [ ]:
#| export
def _doctor_recent_failure_errors():
    errors = []
    recent = [event for event in _failure_data().get("events", [])[-10:] if event.get("kind") == "failure"]
    if recent:
        last = recent[-1]
        errors.append(_warning(
            "recent_tool_failures",
            f"Recent nbskill failure: {last.get('tool')} {last.get('summary') or last.get('error')}",
            "Run doctor(detail='debug', scopes='error') after resolving it.",
            severity="error", scope="error", tool=last.get("tool"),
        ))
    return errors


In [ ]:
#| export
def _doctor_error_items(path, status):
    errors = [
        *_doctor_validation_errors(path),
        *_doctor_order_errors(path),
        *_doctor_recent_failure_errors(),
    ]
    if not status.get("mcp_command_path"):
        errors.append(_warning(
            "mcp_command_missing",
            "nbskill_mcp is not on PATH for this process.",
            "Run uv tool install --editable . --force and reconnect the MCP client.",
            severity="error", scope="error",
        ))
    return errors


In [ ]:
#| export
def _private_symbol_report_text(path):
    return capture_call(private_symbol_report, path=str(path))


In [ ]:
#| export
def _private_symbol_warnings(path):
    try:
        text = _private_symbol_report_text(path)
    except FileNotFoundError as exc:
        text = f"Private symbol scan found a missing notebook: {exc.filename or exc}"
        return [_warning(
            "private_symbol_missing_file",
            text,
            "Remove stale notebook references or recreate the missing notebook before rerunning doctor.",
            severity="warning", scope="warning", path=exc.filename,
        )], text
    if "No cross-notebook private symbol calls found." in text:
        return [], text
    warnings = [
        _warning(
            "private_symbol_call",
            line[2:],
            "Promote the helper to public API or keep the call inside the defining notebook.",
            severity="warning", scope="warning",
        )
        for line in text.splitlines()
        if line.startswith("- ")
    ]
    return warnings, text


In [ ]:
#| export
def _doctor_warning_items(path):
    error_codes = {"exported_py_hash_mismatch", "recent_tool_failures"}
    warnings = [
        {**item, "severity": item.get("severity", "warning"), "scope": "warning"}
        for item in _doctor_warnings(path)
        if item.get("code") not in error_codes
    ]
    private_warnings, private_text = _private_symbol_warnings(path)
    return [*warnings, *private_warnings], private_text


In [ ]:
#| export
def _doctor_style_report(path, skip_folder_re=None, skip_path=None, max_output_chars=12000, max_diagnostics=200):
    chkstyle = run_style_check(path, skip_folder_re, skip_path, strict=False, max_output_chars=max_output_chars)
    try:
        return style_report(path, chkstyle=chkstyle, max_output_chars=max_output_chars, max_diagnostics=max_diagnostics)
    except FileNotFoundError as exc:
        message = f"Style scan found a missing notebook: {exc.filename or exc}"
        diagnostic = {
            "source": "notebook",
            "code": "style_missing_file",
            "severity": "error",
            "path": exc.filename,
            "detail": message,
        }
        output = chkstyle.get("output", "") if isinstance(chkstyle, dict) else ""
        truncated = max_output_chars is not None and len(output) > max_output_chars
        text = output[:max_output_chars].rstrip() if truncated else output.strip()
        if text: text = f"{text}\n\n{message}"
        else: text = message
        return {
            "path": str(path),
            "summary": {
                "notebook_problem_count": 1,
                "chkstyle_problem_count": 0,
                "diagnostic_count": 1,
                "recent_problem_count": 0,
                "output_truncated": truncated,
                "output_chars": len(output),
                "omitted_chars": max(0, len(output) - (max_output_chars or len(output))),
            },
            "diagnostics": [diagnostic],
            "problem_chart": {
                "by_code": {"style_missing_file": 1},
                "by_severity": {"error": 1},
                "by_path": {str(exc.filename): 1} if exc.filename else {},
                "by_source": {"notebook": 1},
            },
            "notebook_problems": [diagnostic],
            "global_usage": {},
            "chkstyle": {"status": chkstyle.get("status", 0) if isinstance(chkstyle, dict) else 0, "text": output[:max_output_chars] if truncated else output, "truncated": truncated, "chars": len(output), "omitted_chars": max(0, len(output) - (max_output_chars or len(output)))},
            "fixes": [],
            "text": text,
        }


In [ ]:
data = _status_data()
assert data["mcp_command"] == "nbskill_mcp"
assert "batch_edit_nb" in data["cli_tools"]

### Registering notebook tools

`create_mcp` is the bridge between this package and an agent client. Each tool is a thin wrapper around a public function, with notebook locks around operations that touch shared files.

The wrapper should preserve useful structure, not flatten everything into text. For example, `symbol_graph` still prints a readable summary, but MCP clients also receive a full `symbol_graph` payload they can inspect without parsing prose.

Edit and execution wrappers also adapt to MCP-specific transport. `update_cell` accepts `new_lines` so exact source can be sent directly without temporary files, and unsafe `exec_nb` runs through a subprocess because the underlying timeout machinery needs the main interpreter thread.

In [ ]:
#| export
_MCP_DIAGNOSTIC_TOOL_CATALOG = {
    'healthcheck': {
        'feature': 'diagnostics',
        'usefulness': 'core',
        'tags': ('status', 'diagnostics', 'setup'),
        'description': 'Cheap liveness probe for the nbskill MCP server, installed version, capabilities, concurrency policy, and reconnect hints.',
        'when_to_use': 'Call first when checking that the MCP server is connected or after reinstalling/exporting tool signatures.',
        'combine_with': 'Could be folded into doctor, but a cheap health probe is useful enough to keep separate.',
    },
    'doctor': {
        'feature': 'diagnostics',
        'usefulness': 'core',
        'tags': ('status', 'diagnostics', 'error', 'warning', 'style'),
        'description': 'Scoped diagnostics for MCP setup, fatal notebook problems, warnings, private symbol leaks, and optional chkstyle output.',
        'when_to_use': "Use scopes='error', scopes='warning', scopes='style', or scopes='all' depending on the diagnostic depth needed.",
        'combine_with': 'Now absorbs private symbol warnings and scoped style diagnostics; healthcheck stays separate as a cheap probe.',
    },
}


In [ ]:
#| export
_MCP_READ_CONTEXT_TOOL_CATALOG = {
    'nb_overview': {
        'feature': 'read_context',
        'usefulness': 'core',
        'tags': ('read', 'notebook', 'orientation'),
        'description': 'Compact notebook map showing Markdown headings, imports, function/class/method signatures, and docstrings; ordinary Markdown docs are optional.',
        'when_to_use': 'Start here when opening a notebook or choosing which chapter or cell to inspect next.',
        'combine_with': 'Do not merge back into a broad reader; this intentionally stays small and scannable.',
    },
    'nb_chapter': {
        'feature': 'read_context',
        'usefulness': 'core',
        'tags': ('read', 'notebook', 'chapter'),
        'description': 'Notebook head plus one selected chapter found by chapter name, text query, or any cell id inside the chapter.',
        'when_to_use': 'Use after nb_overview when a section-level view is enough and line numbers are unnecessary.',
        'combine_with': 'Could share implementation with nb_cell, but the agent-facing context level is distinct.',
    },
    'nb_cell': {
        'feature': 'read_context',
        'usefulness': 'core',
        'tags': ('read', 'notebook', 'cell', 'line-numbers'),
        'description': 'Precise line-numbered cell view; query lookups include previous markdown, examples/tests, and caller/callee usage.',
        'when_to_use': 'Use before editing one cell, especially when line numbers, examples, or usage context matter.',
        'combine_with': 'Keep separate because it is the only reader that should expose line numbers and edit-local context.',
    },
}


In [ ]:
#| export
_MCP_READ_DOC_TOOL_CATALOG = {
    'show_doc': {
        'feature': 'read_context',
        'usefulness': 'situational',
        'tags': ('read', 'symbol', 'documentation'),
        'description': 'Symbol card showing location, nearest docs, signature/docstring, optional source/examples, and compact grouped usage.',
        'when_to_use': 'Use when the task starts from a public symbol and needs API-level context; use nb_cell for line-numbered edit context.',
        'combine_with': 'Could be covered by nb_cell plus symbol search, but it remains useful as a compact API documentation view.',
    },
}


In [ ]:
#| export
_MCP_CELL_EDIT_TOOL_CATALOG = {
    'edit_cell': {
        'feature': 'notebook_edit',
        'usefulness': 'core',
        'tags': ('edit', 'notebook', 'cell', 'source-lines'),
        'description': 'Replace one existing notebook cell from source_lines with validation, optional expected_hash, optional split_lines on empty lines, automatic export, and default-on feedback for exploratory/test cells.',
        'when_to_use': 'Use after nb_cell gives the id. Send source_lines directly; pass split_lines only for empty separator lines. Leave auto_feedback=True unless a speculative edit must not execute.',
        'combine_with': 'Use edit_cell_range for small line edits, insert_cells for new cells, and apply_notebook_edits for coordinated multi-step edits.',
    },
    'edit_cell_range': {
        'feature': 'notebook_edit',
        'usefulness': 'core',
        'tags': ('edit', 'notebook', 'cell', 'line-range'),
        'description': 'Replace a 1-based inclusive line range in one existing notebook cell from replacement_lines, with default-on feedback when the edited cell is exploratory or a test.',
        'when_to_use': 'Use for focused edits inside a large cell when preserving the rest of the cell matters.',
        'combine_with': 'Use edit_cell when replacing the entire cell.',
    },
    'insert_cells': {
        'feature': 'notebook_edit',
        'usefulness': 'core',
        'tags': ('edit', 'notebook', 'insert', 'source-lines'),
        'description': 'Insert one or more notebook cells before or after an anchor id from structured cell objects with source_lines, optional split_lines, and default-on feedback for exploratory/test cells.',
        'when_to_use': 'Use when adding notebook material. Each cell may include cell_type, source_lines, and split_lines.',
        'combine_with': 'Use apply_notebook_edits when inserts must be coordinated with replacements.',
    },
}


In [ ]:
#| export
_MCP_BATCH_EDIT_TOOL_CATALOG = {
    'apply_notebook_edits': {
        'feature': 'notebook_edit',
        'usefulness': 'core',
        'tags': ('edit', 'notebook', 'batch', 'source-lines'),
        'description': 'Apply structured notebook edit operations directly; operations use source_lines/replacement_lines and optional split_lines, never JSON plan text or plan files. Feedback runs by default only for exploratory/test cells.',
        'when_to_use': 'Use for coordinated edits across one or more notebooks when a single tool call is clearer than several direct edit calls.',
        'combine_with': 'This replaces the MCP-facing batch edit plan surface; CLI batch editing remains separate.',
    },
}


In [ ]:
#| export
_MCP_REVIEW_TOOL_CATALOG = {
    'exec_nb': {
        'feature': 'verification',
        'usefulness': 'core',
        'tags': ('execute', 'notebook', 'verify', 'safe'),
        'description': 'Execute a notebook, chapter, or cells up to an id with safe-mode controls and visible output/error capture; unsafe execution is routed through a CLI subprocess so signal-based timeouts run in a main interpreter without blocking MCP worker threads.',
        'when_to_use': 'Use after edits or before trusting notebook behavior; use check_only=True when outputs should not be written.',
        'combine_with': 'Keep separate because execution has distinct safety and concurrency semantics.',
    },
    'diff_nb': {
        'feature': 'review',
        'usefulness': 'core',
        'tags': ('review', 'notebook', 'diff'),
        'description': 'Notebook-aware code-cell diff that avoids raw .ipynb noise and can map generated Python diffs back to notebook owners.',
        'when_to_use': 'Use before final reporting or when reviewing notebook edits without expanding JSON metadata churn.',
        'combine_with': 'Could be grouped with style_check under review, but diff parameters and output are meaningfully different.',
    },
    'style_check': {
        'feature': 'review',
        'usefulness': 'core',
        'tags': ('review', 'style', 'hygiene', 'privacy'),
        'description': 'Notebook hygiene and style report including chkstyle output, private symbol warnings, duplicate imports, stored knowledge warnings, and order issues.',
        'when_to_use': 'Use after substantial edits or when a notebook feels structurally messy; stored behaviour steering regexes are included as warnings.',
        'combine_with': "Doctor can include style diagnostics with scopes='style'; standalone style_check remains the explicit review tool.",
    },
}


In [ ]:
#| export
_MCP_AGENT_TOOL_CATALOG = {
    'execute_plan': {
        'feature': 'agentic_planning',
        'usefulness': 'advanced',
        'tags': ('agent', 'edit', 'plan', 'notebook', 'project'),
        'description': 'Run a bounded edit-interactive plan against one notebook or a project-scoped set of notebooks.',
        'when_to_use': "Use scope='notebook' with notebook=... for one notebook, or scope='project' with notebooks=... for broad plans.",
        'combine_with': 'Combined former execute_project_plan into this tool via scope.',
    },
    'agent_workbench': {
        'feature': 'agentic_planning',
        'usefulness': 'advanced',
        'tags': ('agent', 'context', 'taste', 'contract', 'review'),
        'description': 'Prepare or execute a taste-aware small-diff workbench run with context, budgets, and gates.',
        'when_to_use': 'Use before autonomous implementation when taste, scope, context, and patch budgets need to be explicit.',
        'combine_with': 'Sits above execute_plan; execute_plan remains the bounded notebook executor.',
    },
    'symbol_graph': {
        'feature': 'symbol_analysis',
        'usefulness': 'situational',
        'tags': ('analysis', 'symbol', 'graph'),
        'description': "Analyze one symbol's definitions, callers, caller usage lines, and callees across notebooks.",
        'when_to_use': 'Use when understanding impact, dependencies, or call relationships around one symbol; request JSON/structured content for migration scripts.',
        'combine_with': "Private symbol reporting moved into doctor(scope='warning') and style_check output.",
    },
}


In [ ]:
#| export
_MCP_KNOWLEDGE_TOOL_CATALOG = {
    'store_knowledge': {
        'feature': 'knowledge',
        'usefulness': 'core',
        'tags': ('knowledge', 'memory', 'style', 'regex'),
        'description': 'Store or update one behaviour steering regex and note in the nbskill JSON memory file.',
        'when_to_use': 'Use after translating a remembered good or bad practice into a regex that should warn in future style checks.',
        'combine_with': 'add_behaviour_steering is a shorter regex-only helper; get_knowledge reads stored rules.',
    },
    'add_behaviour_steering': {
        'feature': 'knowledge',
        'usefulness': 'situational',
        'tags': ('knowledge', 'memory', 'regex'),
        'description': 'Add a behaviour steering regex with a generic note.',
        'when_to_use': 'Use when the regex itself is enough context, or prefer store_knowledge when a human-readable note matters.',
        'combine_with': 'store_knowledge is the richer form and style_check applies both kinds of stored rules.',
    },
    'get_knowledge': {
        'feature': 'knowledge',
        'usefulness': 'core',
        'tags': ('knowledge', 'memory', 'lookup'),
        'description': 'Return stored behaviour steering rules, optionally filtered by regex.',
        'when_to_use': 'Use before adding a similar rule, or when inspecting why style_check emitted a stored knowledge warning.',
        'combine_with': 'Use store_knowledge to add or update rules.',
    },
}


In [ ]:
#| export
_MCP_CONVERT_TOOL_CATALOG = {
    'py2nb': {
        'feature': 'conversion',
        'usefulness': 'situational',
        'tags': ('convert', 'python', 'notebook', 'folder'),
        'description': 'Convert one Python file or a folder of Python files into nbdev notebook source with pragmatic cell splitting.',
        'when_to_use': 'Use for file-level or folder-level Python-to-notebook migration.',
        'combine_with': 'Combined former py2nbs behavior into this file-or-folder converter.',
    },
    'py2nbdev': {
        'feature': 'conversion',
        'usefulness': 'situational',
        'tags': ('convert', 'project', 'nbdev'),
        'description': 'Create a pragmatic nbdev project from a pure-Python package or project tree.',
        'when_to_use': 'Use when bootstrapping a whole nbdev project rather than converting one module or folder.',
        'combine_with': 'Keep separate from py2nb because it creates project structure, not only notebooks.',
    },
    'new_nbdev_notebook': {
        'feature': 'conversion',
        'usefulness': 'situational',
        'tags': ('create', 'notebook', 'nbdev'),
        'description': 'Create a minimal nbdev source notebook and exported module.',
        'when_to_use': 'Use when adding a new nbdev notebook/module to a project.',
        'combine_with': 'Use py2nb when converting existing Python source instead of starting a blank notebook.',
    },
}


In [ ]:
#| export
_MCP_TOOL_CATALOG = {
    **_MCP_DIAGNOSTIC_TOOL_CATALOG,
    **_MCP_READ_CONTEXT_TOOL_CATALOG,
    **_MCP_READ_DOC_TOOL_CATALOG,
    **_MCP_CELL_EDIT_TOOL_CATALOG,
    **_MCP_BATCH_EDIT_TOOL_CATALOG,
    **_MCP_REVIEW_TOOL_CATALOG,
    **_MCP_AGENT_TOOL_CATALOG,
    **_MCP_KNOWLEDGE_TOOL_CATALOG,
    **_MCP_CONVERT_TOOL_CATALOG,
}


In [ ]:
#| export
def _mcp_tool_meta(name):
    "Return FastMCP registration metadata for one nbskill tool."
    info = _MCP_TOOL_CATALOG[name]
    return {
        "name": name,
        "description": info["description"],
        "tags": set(info["tags"]),
        "meta": {
            "feature": info["feature"],
            "usefulness": info["usefulness"],
            "when_to_use": info["when_to_use"],
            "combine_with": info["combine_with"],
        },
    }


In [ ]:
#| export
_MCP_CAPABILITIES = ",".join(_MCP_TOOL_CATALOG)
mcp = FastMCP(
    "nbskill",
    instructions=(
        "Work notebook-first in nbdev projects. Feature areas are diagnostics, focused reads, "
        "notebook edits, verification/review, symbol analysis, agentic planning, and conversion. "
        "For reading, use nb_overview for a map, nb_chapter for one section, nb_cell for precise "
        "line-numbered edit context, and show_doc when starting from a public symbol. "
        "For edits, prefer edit_cell for one existing cell, edit_cell_range for partial cell edits, "
        "insert_cells for adding cells, and apply_notebook_edits for coordinated structured edits. Use split_lines only to split at empty source lines. "
        "Edit tools default to auto_feedback=True, which runs only exploratory or test cells in check-only mode and returns their output/errors. "
        "Use exec_nb, diff_nb, and style_check for verification and review; use doctor with "
        "scopes='error', 'warning', 'style', or 'all' for diagnostics. Chkstyle output only appears "
        "when doctor includes the style scope. Reserve execute_plan for agentic notebook/project edits "
        "Use store_knowledge/get_knowledge for behaviour steering memory; style_check applies stored regex rules as warnings. "
        "Normal tool output is concise; use detail='debug' only when troubleshooting. "
        "Notebook operations are concurrency-safe: calls touching the same notebook are serialized, "
        "calls touching different notebooks can run in parallel, and execution uses a global semaphore. "
        "Keep documentation before exported code and show-off examples after it."
    ),
)


In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("healthcheck"))
def _healthcheck_tool(detail: str = "summary") -> ToolResult:
    "Return lightweight nbskill MCP status and point deeper diagnostics to doctor."
    data = _status_data()
    full_output = "\n".join([
        "nbskill mcp ok",
        f"version={data['version']}",
        f"cwd={Path.cwd()}",
        f"python={sys.executable}",
        f"pid={os.getpid()}",
        f"capabilities={_MCP_CAPABILITIES}",
        "parallel=same-notebook operations serialized; different notebooks may run in parallel",
        "execution=global semaphore with one active safe notebook execution",
        "diagnostics=run doctor(scopes='error,warning') for fatal problems and warnings; add style for chkstyle",
        "schema_refresh=restart or reconnect the MCP client after reinstall/export to refresh tool schemas",
    ])
    return mcp_tool_result("healthcheck", {"detail": detail}, full_output, detail=detail, status=data, capabilities=_MCP_CAPABILITIES.split(","))


In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("doctor"))
def doctor_tool(
    path: str = ".", scopes: str = "error,warning", detail: str = "summary",
    fix: bool = False, reset: bool = False, skip_folder_re: str | None = None,
    skip_path: str | None = None, max_output_chars: int = 12000,
    max_diagnostics: int = 200,
) -> ToolResult:
    "Report scoped MCP diagnostics: errors, warnings, and optional chkstyle/style details."
    arguments = dict(path=path, scopes=scopes, detail=detail, fix=fix, reset=reset, skip_folder_re=skip_folder_re, skip_path=skip_path, max_output_chars=max_output_chars, max_diagnostics=max_diagnostics)
    report = _doctor_report(path=path, detail=detail, fix=fix, reset=reset, capabilities=_MCP_CAPABILITIES, scopes=scopes, skip_folder_re=skip_folder_re, skip_path=skip_path, max_output_chars=max_output_chars, max_diagnostics=max_diagnostics)
    return mcp_tool_result(
        "doctor", arguments, report["text"], detail=detail,
        warnings=report["issues"], hints=report["hints"], doctor=report,
    )


In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("nb_overview"))
def _nb_overview_tool(nb_path: str, include_docs: bool = False, detail: str = "summary") -> ToolResult:
    "Show headings, imports, signatures, and docstrings, optionally with non-heading Markdown docs."
    arguments = dict(nb_path=nb_path, include_docs=include_docs, detail=detail)
    full_output = capture_notebook_call(nb_overview, nb_path, nb_path=nb_path, include_docs=include_docs, verbose=True)
    return mcp_tool_result("nb_overview", arguments, full_output, detail=detail)


In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("nb_chapter"))
def _nb_chapter_tool(nb_path: str, query: str | None = None, name: str | None = None, any_cell_id: str | None = None, detail: str = "summary") -> ToolResult:
    "Show the notebook head plus one selected chapter."
    arguments = dict(nb_path=nb_path, query=query, name=name, any_cell_id=any_cell_id, detail=detail)
    full_output = capture_notebook_call(nb_chapter, nb_path, nb_path=nb_path, query=query, name=name, any_cell_id=any_cell_id)
    return mcp_tool_result("nb_chapter", arguments, full_output, detail=detail)


In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("nb_cell"))
def _nb_cell_tool(nb_path: str, query: str | None = None, id: str | None = None, detail: str = "summary") -> ToolResult:
    "Show one selected cell; query lookups include previous docs, examples/tests, and caller/callee usage."
    arguments = dict(nb_path=nb_path, query=query, id=id, detail=detail)
    full_output = capture_notebook_call(nb_cell, nb_path, nb_path=nb_path, query=query, id=id)
    return mcp_tool_result("nb_cell", arguments, full_output, detail=detail)


In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("show_doc"))
def _show_doc_tool(path: str, symbol: str, context: int = 2, source: bool = False, show_ids: bool = False, detail: str = "summary") -> ToolResult:
    "Show a compact symbol card with location, docs, definition, optional source/examples, and grouped usage."
    arguments = dict(path=path, symbol=symbol, context=context, source=source, show_ids=show_ids, detail=detail)
    full_output = capture_notebook_call(show_doc, path, path=path, symbol=symbol, context=context, source=source, show_ids=show_ids)
    return mcp_tool_result("show_doc", arguments, full_output, detail=detail)


In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("edit_cell"))
def edit_cell_tool(
    path: str, cell_id: str, source_lines: list[str], cell_type: str = "code",
    split_lines: list[int] | None = None, validate_code: bool = True,
    expected_hash: str | None = None, auto_feedback: bool = True,
    feedback_timeout: int = 10, feedback_safe: bool = True, detail: str = "summary",
) -> ToolResult:
    "Replace one existing notebook cell from source_lines."
    arguments = dict(path=path, cell_id=cell_id, source_lines=source_lines, cell_type=cell_type, split_lines=split_lines, validate_code=validate_code, expected_hash=expected_hash, auto_feedback=auto_feedback, feedback_timeout=feedback_timeout, feedback_safe=feedback_safe, detail=detail)
    if warning := _mcp_expected_hash_warning(path, cell_id, expected_hash):
        return mcp_tool_result("edit_cell", arguments, warning["message"], detail=detail, warnings=[warning], ok=False)
    start = _mcp_cell_index(path, cell_id)
    cell_count = _mcp_structured_cell_count([dict(source_lines=source_lines, cell_type=cell_type, split_lines=split_lines)], cell_type)
    full_output = replace_notebook_cell(path, cell_id, source_lines, cell_type=cell_type, split_lines=split_lines, validate_code=validate_code, auto_feedback=False)
    cell_ids = _mcp_cell_ids_from_index(path, start, cell_count)
    full_output = _append_mcp_feedback(full_output, path, cell_ids, auto_feedback, feedback_timeout, feedback_safe)
    return mcp_tool_result("edit_cell", arguments, full_output, detail=detail, ok=True, after_hash=_mcp_cell_source_hash(path, cell_id))


In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("edit_cell_range"))
def edit_cell_range_tool(
    path: str, cell_id: str, start_line: int, end_line: int,
    replacement_lines: list[str], validate_code: bool = True,
    expected_hash: str | None = None, auto_feedback: bool = True,
    feedback_timeout: int = 10, feedback_safe: bool = True, detail: str = "summary",
) -> ToolResult:
    "Replace a 1-based inclusive line range in one existing notebook cell."
    arguments = dict(path=path, cell_id=cell_id, start_line=start_line, end_line=end_line, replacement_lines=replacement_lines, validate_code=validate_code, expected_hash=expected_hash, auto_feedback=auto_feedback, feedback_timeout=feedback_timeout, feedback_safe=feedback_safe, detail=detail)
    if warning := _mcp_expected_hash_warning(path, cell_id, expected_hash):
        return mcp_tool_result("edit_cell_range", arguments, warning["message"], detail=detail, warnings=[warning], ok=False)
    full_output = replace_notebook_range(path, cell_id, start_line, end_line, replacement_lines, validate_code=validate_code, auto_feedback=False)
    full_output = _append_mcp_feedback(full_output, path, [cell_id], auto_feedback, feedback_timeout, feedback_safe)
    return mcp_tool_result("edit_cell_range", arguments, full_output, detail=detail, ok=True, after_hash=_mcp_cell_source_hash(path, cell_id))


In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("insert_cells"))
def insert_cells_tool(
    path: str, anchor_id: str, where: str = "after", cells: list[dict] | None = None,
    split_lines: list[int] | None = None, validate_code: bool = True,
    default_cell_type: str = "code", auto_feedback: bool = True,
    feedback_timeout: int = 10, feedback_safe: bool = True, detail: str = "summary",
) -> ToolResult:
    "Insert structured notebook cells before or after an anchor id."
    if where not in {"before", "after"}: raise ValueError("where must be 'before' or 'after'")
    if split_lines is not None:
        if len(cells or []) != 1: raise ValueError("top-level split_lines requires exactly one cell")
        cells = [dict(cells[0], split_lines=split_lines)]
    arguments = dict(path=path, anchor_id=anchor_id, where=where, cells=cells, split_lines=split_lines, validate_code=validate_code, default_cell_type=default_cell_type, auto_feedback=auto_feedback, feedback_timeout=feedback_timeout, feedback_safe=feedback_safe, detail=detail)
    anchor_index = _mcp_cell_index(path, anchor_id)
    start = anchor_index if where == "before" else anchor_index + 1
    cell_count = _mcp_structured_cell_count(cells, default_cell_type)
    full_output = insert_notebook_cells(path, anchor_id, where, cells, validate_code=validate_code, default_cell_type=default_cell_type, auto_feedback=False)
    cell_ids = _mcp_cell_ids_from_index(path, start, cell_count)
    full_output = _append_mcp_feedback(full_output, path, cell_ids, auto_feedback, feedback_timeout, feedback_safe)
    return mcp_tool_result("insert_cells", arguments, full_output, detail=detail, ok=True)


In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("apply_notebook_edits"))
def apply_notebook_edits_tool(
    path: str, edits: list[dict], validate_code: bool = True,
    default_cell_type: str = "code", auto_feedback: bool = True,
    feedback_timeout: int = 10, feedback_safe: bool = True, detail: str = "summary",
) -> ToolResult:
    "Apply structured notebook edit operations without JSON plan text."
    if not edits: raise ValueError("Pass at least one edit")
    arguments = dict(path=path, edits=edits, validate_code=validate_code, default_cell_type=default_cell_type, auto_feedback=auto_feedback, feedback_timeout=feedback_timeout, feedback_safe=feedback_safe, detail=detail)
    warnings = [
        warning for edit in edits
        if (warning := _mcp_expected_hash_warning(str(edit.get("path") or path), edit.get("cell_id"), edit.get("expected_hash")))
    ]
    if warnings:
        return mcp_tool_result("apply_notebook_edits", arguments, "Skipped apply_notebook_edits because one or more expected_hash checks failed.", detail=detail, warnings=warnings, ok=False)
    chunks = []
    for index, edit in enumerate(edits, 1):
        edit_path, start, cell_count = _mcp_feedback_location(path, edit, default_cell_type)
        output = apply_notebook_edit(edit, path, validate_code=validate_code, default_cell_type=default_cell_type, auto_feedback=False)
        cell_ids = _mcp_cell_ids_from_index(edit_path, start, cell_count)
        output = _append_mcp_feedback(output, edit_path, cell_ids, auto_feedback, feedback_timeout, feedback_safe)
        chunks.append(f"#{index} {edit.get('op')}\n{output}".rstrip())
    return mcp_tool_result("apply_notebook_edits", arguments, "\n\n".join(chunks), detail=detail, ok=True)
    return mcp_tool_result("apply_notebook_edits", arguments, "\n\n".join(chunks), detail=detail, ok=True)


In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("exec_nb"))
def exec_nb_tool(
    path: str, dest: str | None = None, exc_stop: bool = False, up2id: int | str | None = None,
    chapter: str | None = None, timeout: int = 30, show_output: bool = True,
    verbose: bool = False, safe: bool = True, allow: str | None = None,
    ok_dests: str | None = None, cache_httpx: bool = False, cache_dir: str | None = None,
    cache_domains: str | None = None, allow_new: bool = False, check_only: bool = False,
    detail: str = "summary",
) -> ToolResult:
    "Execute a notebook and return visible outputs/errors. Use check_only=True to avoid writing outputs."
    arguments = dict(path=path, dest=dest, exc_stop=exc_stop, up2id=up2id, chapter=chapter, timeout=timeout, show_output=show_output, verbose=verbose, safe=safe, allow=allow, ok_dests=ok_dests, cache_httpx=cache_httpx, cache_dir=cache_dir, cache_domains=cache_domains, allow_new=allow_new, check_only=check_only, detail=detail)
    if safe:
        full_output = capture_notebook_call(exec_nb, path, dest or path, **{k: v for k, v in arguments.items() if k != "detail"})
    else:
        full_output = _capture_exec_nb_cli_call(arguments)
    warnings = []
    if safe and "PermissionError: Audit:" in full_output:
        warnings.append(_warning(
            "safe-exec-audit-block",
            "Safe execution blocked an audited operation.",
            "For trusted notebooks, rerun with safe=False.",
        ))
    return mcp_tool_result("exec_nb", arguments, full_output, detail=detail, warnings=warnings)


In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("diff_nb"))
def _diff_nb_tool(path: str, ref_a: str | None = "HEAD", ref_b: str | None = None, adds: bool = True, changes: bool = True, dels: bool = False, cell_id: str | None = None, after_id: str | None = None, show_owner: bool = False, detail: str = "summary") -> ToolResult:
    "Diff notebook code cells without expanding raw notebook JSON; optionally filter by cell id or map generated Python to its owner."
    arguments = dict(path=path, ref_a=ref_a, ref_b=ref_b, adds=adds, changes=changes, dels=dels, cell_id=cell_id, after_id=after_id, show_owner=show_owner, detail=detail)
    if show_owner and Path(path).suffix == ".py":
        return mcp_tool_result("diff_nb", arguments, _owner_output(path), detail=detail)
    full_output = capture_notebook_call(diff_nb, path, **{k: v for k, v in arguments.items() if k not in {"show_owner", "detail"}})
    return mcp_tool_result("diff_nb", arguments, full_output, detail=detail)


In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("execute_plan"))
def execute_plan_tool(
    plan: str, notebook: str | None = None, scope: str = "notebook",
    notebooks: str | None = None, model: str | None = None, max_steps: int = 20,
    timeout: int = 30,
    detail: str = "summary",
) -> ToolResult:
    "Run a bounded edit-interactive loop against one notebook or a project notebook set."
    arguments = dict(plan=plan, notebook=notebook, scope=scope, notebooks=notebooks, model=model, max_steps=max_steps, timeout=timeout, detail=detail)
    mode = "project" if scope == "project" or notebooks else "notebook"
    if mode == "project":
        project_args = dict(plan=plan, notebooks=notebooks, model=model, max_steps=max_steps, timeout=timeout, dry_run=False)
        full_output = capture_call(execute_project_plan, **project_args)
        return mcp_tool_result("execute_plan", arguments, full_output, detail=detail)
    if not notebook: raise ValueError("notebook is required when scope='notebook'")
    notebook_args = dict(notebook=notebook, plan=plan, model=model, max_steps=max_steps, timeout=timeout, dry_run=False)
    result = execute_plan(**notebook_args)
    full_output = plan_result_text(result)
    tool_result = mcp_tool_result("execute_plan", arguments, full_output, detail=detail)
    if isinstance(result, dict):
        tool_result.structured_content["history"] = result.get("history", [])
        tool_result.structured_content["summary"] = result.get("summary", "")
        tool_result.structured_content["execute_plan"] = result
    return tool_result


In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("agent_workbench"))
def agent_workbench_tool(
    goal: str, notebook: str | None = None, contract_file: str | None = None,
    execute: bool = False, max_steps: int = 8, timeout: int = 30,
    detail: str = "summary",
) -> ToolResult:
    "Prepare or execute a taste-aware small-diff workbench run."
    arguments = dict(goal=goal, notebook=notebook, contract_file=contract_file, execute=execute, max_steps=max_steps, timeout=timeout, detail=detail)
    result = agent_workbench_result(
        goal, notebook=notebook, contract_file=contract_file, execute=execute,
        max_steps=max_steps, timeout=timeout,
    )
    full_output = result.get("rendered_plan") or result.get("summary", "")
    tool_result = mcp_tool_result("agent_workbench", arguments, full_output, detail=detail)
    if isinstance(result, dict): tool_result.structured_content["agent_workbench"] = result
    return tool_result


In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("symbol_graph"))
def _symbol_graph_tool(path: str = "nbs", symbol: str = "", json_output: bool = False, detail: str = "summary") -> ToolResult:
    "Show definitions, callers, and callees for one notebook symbol."
    arguments = dict(path=path, symbol=symbol, json_output=json_output, detail=detail)
    full_output = capture_call(symbol_graph, path=path, symbol=symbol, json_output=json_output)
    result = mcp_tool_result("symbol_graph", arguments, full_output, detail=detail)
    if symbol:
        result.structured_content["symbol_graph"] = symbol_graph_public_data(symbol_graph_data(path, symbol))
    return result


In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("style_check"))
def style_check_tool(
    path: str = ".", skip_folder_re: str | None = None, skip_path: str | None = None,
    strict: bool = False, delete_after_output: bool = False,
    max_output_chars: int = 12000, max_diagnostics: int = 200, fix: bool = False,
    changed_only: bool = False, ref_a: str | None = "HEAD", ref_b: str | None = None,
    detail: str = "summary",
) -> ToolResult:
    "Print capped chkstyle output, notebook hygiene warnings, private symbol warnings, and global tool usage."
    arguments = dict(path=path, skip_folder_re=skip_folder_re, skip_path=skip_path, strict=strict, delete_after_output=delete_after_output, max_output_chars=max_output_chars, max_diagnostics=max_diagnostics, fix=fix, changed_only=changed_only, ref_a=ref_a, ref_b=ref_b, detail=detail)
    style_output = capture_call(style_check, **{k: v for k, v in arguments.items() if k != "detail"})
    private_output = _private_symbol_report_text(path)
    full_output = "\n\n".join(chunk for chunk in [private_output, style_output] if chunk)
    report = style_report(path, max_output_chars=max_output_chars, max_diagnostics=max_diagnostics, changed_only=changed_only, ref_a=ref_a, ref_b=ref_b)
    report["private_symbol_report"] = private_output
    return mcp_tool_result("style_check", arguments, full_output, max_output_chars=max_output_chars, detail=detail, style_report=report)


In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("store_knowledge"))
def _store_knowledge_tool(apply_regex: str, note: str, path: str | None = None, detail: str = "summary") -> ToolResult:
    "Store or update one behaviour steering regex and note."
    arguments = dict(apply_regex=apply_regex, note=note, path=path, detail=detail)
    full_output = capture_call(store_knowledge, apply_regex=apply_regex, note=note, path=path)
    return mcp_tool_result("store_knowledge", arguments, full_output, detail=detail)


In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("add_behaviour_steering"))
def _add_behaviour_steering_tool(regex: str, path: str | None = None, detail: str = "summary") -> ToolResult:
    "Add a behaviour steering regex with a generic note."
    arguments = dict(regex=regex, path=path, detail=detail)
    full_output = capture_call(add_behaviour_steering, regex=regex, path=path)
    return mcp_tool_result("add_behaviour_steering", arguments, full_output, detail=detail)


In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("get_knowledge"))
def _get_knowledge_tool(regex: str | None = None, path: str | None = None, detail: str = "summary") -> ToolResult:
    "Return stored behaviour steering rules, optionally filtered by regex."
    arguments = dict(regex=regex, path=path, detail=detail)
    full_output = capture_call(get_knowledge, regex=regex, path=path)
    return mcp_tool_result("get_knowledge", arguments, full_output, detail=detail)


In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("py2nb"))
def py2nb_tool(
    path: str, nbs_path: str = "nbs", dest: str | None = None, recursive: bool = True,
    maxdepth: int | None = None, preserve_tree: bool = True, class_lines: int = 100,
    method_lines: int = 10, package: str | None = None, include: str | None = None,
    exclude: str | None = None, skip_init: bool = True, include_tests: bool = False,
    force: bool = True, detail: str = "summary",
) -> ToolResult:
    "Convert one Python file or a folder of Python files into nbdev notebook source."
    arguments = dict(path=path, nbs_path=nbs_path, dest=dest, recursive=recursive, maxdepth=maxdepth, preserve_tree=preserve_tree, class_lines=class_lines, method_lines=method_lines, package=package, include=include, exclude=exclude, skip_init=skip_init, include_tests=include_tests, force=force, detail=detail)
    full_output = capture_call(py2nb, **{k: v for k, v in arguments.items() if k != "detail"}, dry_run=False)
    return mcp_tool_result("py2nb", arguments, full_output, detail=detail)


In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("py2nbdev"))
def _py2nbdev_tool(source: str, dest: str, package: str | None = None, nbs_path: str = "nbs", force: bool = False, run_validation: bool = True, detail: str = "summary") -> ToolResult:
    "Create a pragmatic nbdev project from a pure-Python package."
    arguments = dict(source=source, dest=dest, package=package, nbs_path=nbs_path, force=force, run_validation=run_validation, detail=detail)
    full_output = capture_call(py2nbdev, **{k: v for k, v in arguments.items() if k != "detail"}, dry_run=False)
    return mcp_tool_result("py2nbdev", arguments, full_output, detail=detail)


In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("new_nbdev_notebook"))
def new_nbdev_notebook_tool(
    name: str, default_exp: str | None = None, title: str | None = None,
    nbs_path: str = "nbs", force: bool = False, detail: str = "summary",
) -> ToolResult:
    "Create a minimal nbdev source notebook and exported module."
    arguments = dict(name=name, default_exp=default_exp, title=title, nbs_path=nbs_path, force=force, detail=detail)
    full_output = capture_call(new_nbdev_notebook, **{k: v for k, v in arguments.items() if k != "detail"}, dry_run=False)
    return mcp_tool_result("new_nbdev_notebook", arguments, full_output, detail=detail)


In [ ]:

#| export
def create_mcp():
    "Return the public nbskill FastMCP server."
    return mcp


### Running the server

The CLI entry point only chooses the transport and starts FastMCP. Keeping startup separate from tool registration makes `create_mcp` easy to test without launching a long-running server.

In [ ]:
#| export
@call_parse
def main(
    transport: str = "stdio",  # MCP transport; stdio is what Codex/Claude use for local servers
    show_banner: bool = False,  # Show FastMCP startup banner
):
    "Run the nbskill MCP server."
    mcp.run(transport=transport, show_banner=show_banner)

### MCP transport-friendly edits

MCP clients should be able to send exact source directly. `source_lines` avoids JSON-string plans, temporary files, and CLI newline decoding entirely, which is useful when the source being written contains escaped notebook text. Unsafe notebook execution is still available through MCP, but it runs in a subprocess so signal-based timeouts stay in a main interpreter.

In [ ]:
mcp_demo = create_mcp()
mcp_tools = {tool.name: tool for tool in await mcp_demo.list_tools()}
assert "source_lines" in str(mcp_tools["edit_cell"].parameters)
assert "split_lines" in str(mcp_tools["edit_cell"].parameters)
assert "replacement_lines" in str(mcp_tools["edit_cell_range"].parameters)
assert "auto_feedback" in str(mcp_tools["edit_cell"].parameters)
print("direct MCP schema exposes structured edit lines")

direct MCP schema exposes structured edit lines


In [ ]:
with write_demo_notebook("07_mcp_split_lines.ipynb") as edit_nb:
    nb = new_nb([mk_cell('payload = "old"\nother = 1')])
    edit_cell_id = nb.cells[0].id
    _write_raw_nb(nb, edit_nb)
    split_source = ["left = 1", "", "right = 2"]
    split_result = await mcp_demo.call_tool("edit_cell", {"path": str(edit_nb), "cell_id": edit_cell_id, "source_lines": split_source, "split_lines": [1, 2]})
    assert "into 2 cells" in split_result.structured_content["full_output"]
    assert [cell.source for cell in _read_raw_nb(edit_nb).cells[:2]] == ["left = 1", "right = 2"]
    print("direct MCP split_lines split only on empty lines")

direct MCP split_lines split only on empty lines


In [ ]:
with write_demo_notebook("07_mcp_source_lines.ipynb") as edit_nb:
    nb = new_nb([mk_cell('payload = "old"\nother = 1')])
    edit_cell_id = nb.cells[0].id
    _write_raw_nb(nb, edit_nb)
    exact_source = 'payload = "line 1' + chr(92) + 'nline 2"'
    edit_result = await mcp_demo.call_tool("edit_cell", {"path": str(edit_nb), "cell_id": edit_cell_id, "source_lines": [exact_source]})
    assert "Updated cell" in edit_result.structured_content["full_output"]
    assert _read_raw_nb(edit_nb).cells[0].source == exact_source
    range_result = await mcp_demo.call_tool("edit_cell_range", {"path": str(edit_nb), "cell_id": edit_cell_id, "start_line": 1, "end_line": 1, "replacement_lines": ['payload = "range"']})
    assert "Updated lines 1:1" in range_result.structured_content["full_output"]
    feedback_result = await mcp_demo.call_tool("edit_cell", {"path": str(edit_nb), "cell_id": edit_cell_id, "source_lines": ["print('mcp feedback')"]})
    assert "Auto feedback" in feedback_result.structured_content["full_output"]
    assert "mcp feedback" in feedback_result.structured_content["full_output"]
    assert _read_raw_nb(edit_nb).cells[0].outputs == []
    quiet_result = await mcp_demo.call_tool("edit_cell", {"path": str(edit_nb), "cell_id": edit_cell_id, "source_lines": ["payload = 1"], "auto_feedback": False})
    assert "Auto feedback" not in quiet_result.structured_content["full_output"]
    print("direct MCP edit tools preserved escaped newlines and returned subprocess feedback")

direct MCP edit tools preserved escaped newlines and returned subprocess feedback


In [ ]:
with write_demo_notebook("07_mcp_exec_nb.ipynb") as unsafe_nb:
    _write_raw_nb(new_nb([mk_cell("print('unsafe mcp')")]), unsafe_nb)
    exec_result = await mcp_demo.call_tool("exec_nb", {"path": str(unsafe_nb), "safe": False, "allow_new": True, "timeout": 5})
    assert "unsafe mcp" in exec_result.structured_content["full_output"]
    print("unsafe MCP exec_nb used subprocess without signal errors")

unsafe MCP exec_nb used subprocess without signal errors


In [ ]:
mcp = create_mcp()


In [ ]:
tools = {tool.name: tool for tool in await mcp.list_tools()}


In [ ]:
assert {
    "healthcheck", "doctor", "nb_overview", "nb_chapter", "nb_cell",
    "edit_cell", "edit_cell_range", "insert_cells", "apply_notebook_edits",
    "exec_nb", "show_doc", "execute_plan", "agent_workbench", "symbol_graph",
    "style_check", "py2nb", "py2nbdev",
} <= set(tools)


In [ ]:
assert "write_nb" not in tools


In [ ]:
assert "update_cell" not in tools


In [ ]:
assert "batch_edit_nb" not in tools


In [ ]:
assert "execute_project_plan" not in tools


In [ ]:
assert "private_symbol_report" not in tools


In [ ]:
assert "py2nbs" not in tools


In [ ]:
assert "read_nb" not in tools


In [ ]:
assert "include_markdown" not in str(tools["nb_overview"].parameters)


In [ ]:
assert "show_ids" not in str(tools["nb_overview"].parameters)


In [ ]:
assert "show_ids" not in str(tools["nb_chapter"].parameters)


In [ ]:
assert "show_ids" not in str(tools["nb_cell"].parameters)


In [ ]:
assert "include_docs" in str(tools["nb_overview"].parameters)


In [ ]:
assert "verbose" not in str(tools["nb_overview"].parameters)


In [ ]:
assert "detail" in str(tools["nb_cell"].parameters)


In [ ]:
assert "show_owner" in str(tools["diff_nb"].parameters)


In [ ]:
assert "cell_id" in str(tools["diff_nb"].parameters)


In [ ]:
assert "after_id" in str(tools["diff_nb"].parameters)


In [ ]:
assert "check_only" in str(tools["exec_nb"].parameters)


In [ ]:
assert "scope" in str(tools["execute_plan"].parameters)


In [ ]:
assert "notebooks" in str(tools["execute_plan"].parameters)


In [ ]:
assert "source_lines" in str(tools["edit_cell"].parameters)


In [ ]:
assert "split_lines" in str(tools["edit_cell"].parameters)


In [ ]:
assert "auto_feedback" in str(tools["edit_cell"].parameters)


In [ ]:
assert "feedback_timeout" in str(tools["apply_notebook_edits"].parameters)


In [ ]:
assert "split_lines" in str(tools["insert_cells"].parameters)


In [ ]:
assert "replacement_lines" in str(tools["edit_cell_range"].parameters)


In [ ]:
assert "anchor_id" in str(tools["insert_cells"].parameters)


In [ ]:
assert "edits" in str(tools["apply_notebook_edits"].parameters)


In [ ]:
assert "max_output_chars" in str(tools["style_check"].parameters)


In [ ]:
assert "changed_only" in str(tools["style_check"].parameters)


In [ ]:
assert "scopes" in str(tools["doctor"].parameters)


In [ ]:
assert "json_output" in str(tools["symbol_graph"].parameters)


In [ ]:
assert "delete_after_outout" not in str(tools["style_check"].parameters)


In [ ]:
for name in ("edit_cell", "edit_cell_range", "insert_cells", "apply_notebook_edits", "execute_plan", "style_check", "py2nb", "py2nbdev"):
    assert "dry_run" not in str(tools[name].parameters)


In [ ]:
assert "Compact notebook map" in tools["nb_overview"].description


In [ ]:
assert "line-numbered" in tools["nb_cell"].description


In [ ]:
assert {"read", "notebook", "cell"} <= set(tools["nb_cell"].tags)


In [ ]:
assert {"edit", "notebook", "cell"} <= set(tools["edit_cell"].tags)


In [ ]:
assert {"edit", "notebook", "line-range"} <= set(tools["edit_cell_range"].tags)


In [ ]:
assert "source_lines" in tools["edit_cell"].description


In [ ]:
assert "replacement_lines" in tools["edit_cell_range"].description


In [ ]:
assert "structured notebook edit operations" in tools["apply_notebook_edits"].description


In [ ]:
assert tools["execute_plan"].meta["feature"] == "agentic_planning"


In [ ]:
assert "Combined former execute_project_plan" in tools["execute_plan"].meta["combine_with"]


In [ ]:
assert tools["agent_workbench"].meta["feature"] == "agentic_planning"


In [ ]:
assert "execute" in str(tools["agent_workbench"].parameters)


In [ ]:
assert tools["py2nb"].meta["usefulness"] == "situational"


In [ ]:
assert "Combined former py2nbs" in tools["py2nb"].meta["combine_with"]


In [ ]:
assert "caller usage lines" in tools["symbol_graph"].description


In [ ]:
assert "CLI subprocess" in tools["exec_nb"].description


In [ ]:
graph_root = demo_path("07_mcp_symbol_graph")


In [ ]:
try:
    graph_root.mkdir()
    lib_nb = graph_root / "lib.ipynb"
    call_nb = graph_root / "call.ipynb"
    _write_raw_nb(new_nb([mk_cell("#| export\ndef thing():\n    return 1")]), lib_nb)
    _write_raw_nb(new_nb([mk_cell("from nbskill.lib import thing\nvalue = thing()")]), call_nb)
    symbol_result = await mcp.call_tool("symbol_graph", {"path": str(graph_root), "symbol": "thing", "json_output": True})
    assert symbol_result.structured_content["symbol_graph"]["caller_usages"][0]["line"] == "value = thing()"
    print("structured symbol_graph MCP usages:", len(symbol_result.structured_content["symbol_graph"]["caller_usages"]))
finally:
    remove_demo_path(graph_root)


In [ ]:
edit_root = demo_path("07_mcp_direct_update")


In [ ]:
async def _exercise_mcp_edit_tools(edit_root):
    edit_root.mkdir()
    edit_nb = edit_root / "edit.ipynb"
    nb = new_nb([mk_cell('payload = "old"\nvalue = 1')])
    cell_id = nb.cells[0].id
    _write_raw_nb(nb, edit_nb)
    exact_source = 'payload = "line 1' + chr(92) + 'nline 2"'
    edit_result = await mcp.call_tool("edit_cell", {"path": str(edit_nb), "cell_id": cell_id, "source_lines": [exact_source]})
    assert "Updated cell" in edit_result.structured_content["full_output"]
    assert _read_raw_nb(edit_nb).cells[0].source == exact_source

    range_result = await mcp.call_tool("edit_cell_range", {"path": str(edit_nb), "cell_id": cell_id, "start_line": 1, "end_line": 1, "replacement_lines": ['payload = "range"']})
    assert "Updated lines 1:1" in range_result.structured_content["full_output"]
    assert _read_raw_nb(edit_nb).cells[0].source == 'payload = "range"'

    insert_result = await mcp.call_tool("insert_cells", {"path": str(edit_nb), "anchor_id": cell_id, "where": "after", "cells": [{"cell_type": "code", "source_lines": ["inserted = True"]}]})
    assert "Inserted 1 cell" in insert_result.structured_content["full_output"]
    assert _read_raw_nb(edit_nb).cells[1].source == "inserted = True"

    batch_nb = edit_root / "batch.ipynb"
    batch = new_nb([mk_cell('payload = "old"')])
    batch_id = batch.cells[0].id
    _write_raw_nb(batch, batch_nb)
    batch_result = await mcp.call_tool("apply_notebook_edits", {"path": str(batch_nb), "edits": [{"op": "replace_cell", "cell_id": batch_id, "source_lines": ['payload = "new"']}]})
    assert "Updated cell" in batch_result.structured_content["full_output"]
    assert _read_raw_nb(batch_nb).cells[0].source == 'payload = "new"'


In [ ]:
edit_root = demo_path("07_mcp_edit_tools")
try:
    await _exercise_mcp_edit_tools(edit_root)
    print("direct MCP edit tools used source_lines without JSON plan text")
finally:
    remove_demo_path(edit_root)


In [ ]:
edit_root = demo_path("07_mcp_unsafe_exec")
try:
    edit_root.mkdir()
    unsafe_nb = edit_root / "unsafe.ipynb"
    _write_raw_nb(new_nb([mk_cell("print('unsafe mcp')")]), unsafe_nb)
    exec_result = await mcp.call_tool("exec_nb", {"path": str(unsafe_nb), "safe": False, "allow_new": True, "timeout": 5})
    assert "unsafe mcp" in exec_result.structured_content["full_output"]
    print("unsafe MCP exec_nb ran through CLI subprocess")
finally:
    remove_demo_path(edit_root)


In [ ]:
assert _doctor_scope_set("all") == {"error", "warning", "style"}


In [ ]:
doctor = _doctor_report(".", scopes="error,warning")


In [ ]:
assert doctor["style"] is None


In [ ]:
assert "errors" in doctor


In [ ]:
assert "warnings" in doctor


In [ ]:
assert "status" in doctor


In [ ]:
style_doctor = _doctor_report(".", scopes="style", max_output_chars=200, max_diagnostics=5)


In [ ]:
assert style_doctor["style"] is not None


In [ ]:
assert "chkstyle" in style_doctor["style"]


In [ ]:
redacted = mcp_tool_result(
    "edit_cell",
    {"path": "nbs/example.ipynb", "cell_id": "abc123", "source_lines": ["x" * 1000]},
    "ok",
)


In [ ]:
assert "x" * 200 not in redacted.structured_content["summary"]


In [ ]:
assert redacted.structured_content["call"]["arguments"]["source_lines"].startswith("<1004 chars redacted")


In [ ]:
debug = mcp_tool_result("nb_cell", {"nb_path": "nbs/example.ipynb", "id": "abc123"}, "ok", detail="debug")


In [ ]:
assert debug.structured_content["debug"]["arguments"] == {"nb_path": "nbs/example.ipynb", "id": "abc123"}


In [ ]:
calls = []


In [ ]:
old_doctor_warnings = _doctor_warnings


In [ ]:
def _capture_mcp_warning_scopes():
    try:
        def fake_doctor_warnings(path=".", scope_path=None, scope_cell_ids=None):
            cell_scope = None if scope_cell_ids is None else set(scope_cell_ids)
            calls.append((path, scope_path, cell_scope))
            return []

        globals()["_doctor_warnings"] = fake_doctor_warnings
        mcp_tool_result("healthcheck", {}, "ok")
        mcp_tool_result("nb_overview", {"nb_path": "nbs/overview.ipynb"}, "Cell id=ignored")
        mcp_tool_result("nb_chapter", {"nb_path": "nbs/chapter.ipynb"}, "Cell id=ch1\nCell id=ch2")
        mcp_tool_result("nb_cell", {"nb_path": "nbs/cell.ipynb", "id": "abc123"}, "Cell id=abc123\nCaller usages:\n- nbs/cell.ipynb id=caller456 line 1: call()")
        mcp_tool_result("show_doc", {"path": "nbs/symbol.ipynb", "symbol": "thing"}, "Location: nbs/symbol.ipynb Cell id=def123: code\nCaller usages:\n- nbs/symbol.ipynb id=caller789 line 1: thing()")
        mcp_tool_result("diff_nb", {"path": "nbs/07_mcp.ipynb"}, "diff without cell ids")
        mcp_tool_result("diff_nb", {"path": "nbs/07_mcp.ipynb"}, "--- code cell abc123 ---\nchanged")
        mcp_tool_result("insert_cells", {"path": "nbs/write_scope.ipynb", "anchor_id": "anchor123"}, "Wrote 3 cells to nbs/write_scope.ipynb")
        mcp_tool_result("edit_cell", {"path": "nbs/update_scope.ipynb", "cell_id": "upd123"}, "Updated cell id=upd123")
        mcp_tool_result(
            "apply_notebook_edits",
            {"path": "nbs/batch_a.ipynb", "edits": [
                {"op": "replace_cell", "path": "nbs/batch_a.ipynb", "cell_id": "a1", "source_lines": ["a"]},
                {"op": "replace_cell", "path": "nbs/batch_b.ipynb", "cell_id": "b1", "source_lines": ["b"]},
            ]},
            "Updated cell id=a1\nUpdated cell id=b1",
        )
    finally:
        globals()["_doctor_warnings"] = old_doctor_warnings


In [ ]:
_capture_mcp_warning_scopes()


In [ ]:
assert calls == [
    ("nbs/overview.ipynb", "nbs/overview.ipynb", None),
    ("nbs/chapter.ipynb", "nbs/chapter.ipynb", {"ch1", "ch2"}),
    ("nbs/cell.ipynb", "nbs/cell.ipynb", {"abc123"}),
    ("nbs/symbol.ipynb", "nbs/symbol.ipynb", {"def123"}),
    ("nbs/07_mcp.ipynb", "nbs/07_mcp.ipynb", set()),
    ("nbs/07_mcp.ipynb", "nbs/07_mcp.ipynb", {"abc123"}),
    ("nbs/write_scope.ipynb", "nbs/write_scope.ipynb", {"anchor123"}),
    ("nbs/update_scope.ipynb", "nbs/update_scope.ipynb", {"upd123"}),
    ("nbs/batch_a.ipynb", "nbs/batch_a.ipynb", {"a1", "b1"}),
    ("nbs/batch_b.ipynb", "nbs/batch_b.ipynb", {"a1", "b1"}),
]


In [ ]:
source_calls = []


In [ ]:
root = _git_root(".")


In [ ]:
old_generated_pairs = _generated_pairs_for_scope


In [ ]:
old_validation_problems = notebook_validation_problems


In [ ]:
old_style_warnings = _style_problem_warnings


In [ ]:
old_doc_warnings = _doc_script_warnings


In [ ]:
old_failure_data = _failure_data


In [ ]:
try:
    def fake_generated_pairs(root, path):
        source_calls.append(("generated", Path(path)))
        return []

    def fake_validation_problems(path):
        source_calls.append(("validation", Path(path)))
        return []

    def fake_style_warnings(path, root):
        source_calls.append(("style", Path(path)))
        return []

    def fake_doc_warnings(root):
        source_calls.append(("docs", Path(root)))
        return []

    def fake_failure_data():
        source_calls.append(("failure", None))
        return {"events": []}

    _generated_pairs_for_scope = fake_generated_pairs
    notebook_validation_problems = fake_validation_problems
    _style_problem_warnings = fake_style_warnings
    _doc_script_warnings = fake_doc_warnings
    _failure_data = fake_failure_data
    _doctor_warnings("nbs/cell.ipynb", scope_path="nbs/cell.ipynb")
    _doctor_warnings(".")
finally:
    _generated_pairs_for_scope = old_generated_pairs
    notebook_validation_problems = old_validation_problems
    _style_problem_warnings = old_style_warnings
    _doc_script_warnings = old_doc_warnings
    _failure_data = old_failure_data


In [ ]:
assert source_calls == [
    ("generated", root / "nbs/cell.ipynb"),
    ("validation", root / "nbs/cell.ipynb"),
    ("style", root / "nbs/cell.ipynb"),
    ("generated", root),
    ("validation", root),
    ("style", root),
    ("docs", root),
    ("failure", None),
]


In [ ]:
sample = [
    _warning("other", "other notebook", path="nbs/other.ipynb"),
    _warning("same_other_cell", "same notebook other cell", path="nbs/07_mcp.ipynb", cell_id="other-cell"),
    _warning("current", "current displayed cell", path="nbs/07_mcp.ipynb", cell_id="abc123"),
    _warning("notebook_export_missing", "current notebook", path="nbs/07_mcp.ipynb"),
]


In [ ]:
filtered = _filter_warnings_for_scope(sample, Path(".").resolve(), "nbs/07_mcp.ipynb", {"abc123"})


In [ ]:
assert [item["code"] for item in filtered] == ["current", "notebook_export_missing"]


In [ ]:
assert not _cell_export_relevant({"cell_type": "code", "source": ["print('demo')"]})


In [ ]:
assert _cell_export_relevant({"cell_type": "code", "source": ["#| export\ndef exported():\n    pass"]})


In [ ]:
print("notebook-only edits do not require generated-file warnings")


In [ ]:
module_path = Path("nbskill/mcp.py")


In [ ]:
if not module_path.exists(): module_path = Path("../nbskill/mcp.py")


In [ ]:
owner = _generated_owner(module_path)


In [ ]:
assert owner and owner.name == "07_mcp.ipynb"


In [ ]:
assert "07_mcp.ipynb" in _owner_output(module_path)


In [ ]:
assert _git_root("nbs/07_mcp.ipynb") == _git_root(".")


In [ ]:
assert _rel_to_root("nbs/07_mcp.ipynb", _git_root(".")) == "nbs/07_mcp.ipynb"


In [ ]:
root = demo_path("07_mcp_insert_lines")
try:
    root.mkdir()
    path = root / "example.ipynb"
    nb = new_nb([mk_cell("anchor = True")])
    anchor_id = nb.cells[0].id
    _write_raw_nb(nb, path)
    mcp = _mcp_mod.create_mcp()
    result = await mcp.call_tool(
        "insert_cells",
        {
            "path": str(path),
            "anchor_id": anchor_id,
            "cells": [{"cell_type": "code", "source_lines": ['source = "line 1\\nline 2"']}],
        },
    )
    assert "Inserted 1 cell" in str(result)
    assert _read_raw_nb(path).cells[1].source == 'source = "line 1\\nline 2"'
finally:
    remove_demo_path(root)

In [ ]:
calls = {}


In [ ]:
project_calls = {}


In [ ]:
old_execute_plan = _mcp_mod.execute_plan


In [ ]:
old_execute_project_plan = _mcp_mod.execute_project_plan


In [ ]:
async def _exercise_agent_mcp_tools():
    try:
        def fake_execute_plan(**kwargs):
            calls.update(kwargs)
            return {"summary": "delegated summary", "history": [{"tool": "add_cell"}], "text": "delegated"}

        def fake_execute_project_plan(**kwargs):
            project_calls.update(kwargs)
            print("project delegated")
            return {"summary": "project summary"}

        _mcp_mod.execute_plan = fake_execute_plan
        _mcp_mod.execute_project_plan = fake_execute_project_plan
        server = _mcp_mod.create_mcp()
        result = await server.call_tool("execute_plan", {"notebook": "nbs/index.ipynb", "plan": "noop", "model": "fake", "max_steps": 1, "timeout": 2})
        assert calls == {"notebook": "nbs/index.ipynb", "plan": "noop", "model": "fake", "max_steps": 1, "timeout": 2, "dry_run": False}
        assert "delegated" in str(result)
        assert result.structured_content["summary"] == "delegated summary"
        assert result.structured_content["history"] == [{"tool": "add_cell"}]

        project = await server.call_tool("execute_plan", {"scope": "project", "notebooks": "nbs/01_read.ipynb,nbs/02_write.ipynb", "plan": "noop", "model": "fake", "max_steps": 1, "timeout": 2})
        assert project_calls == {"plan": "noop", "notebooks": "nbs/01_read.ipynb,nbs/02_write.ipynb", "model": "fake", "max_steps": 1, "timeout": 2, "dry_run": False}
        assert "project delegated" in str(project)

        project_root = Path.cwd().parent if Path.cwd().name == "nbs" else Path.cwd()
        token = _mcp_mod._in_call_parse.set(True)
        try:
            workbench = await server.call_tool("agent_workbench", {"goal": "touch nothing", "notebook": str(project_root / "nbs/11_agent_workbench.ipynb")})
        finally:
            _mcp_mod._in_call_parse.reset(token)
        assert "Agent workbench task" in str(workbench)
        assert workbench.structured_content["agent_workbench"]["summary"] == "agent_workbench prepared execution context"
    finally:
        _mcp_mod.execute_plan = old_execute_plan
        _mcp_mod.execute_project_plan = old_execute_project_plan


In [ ]:
await _exercise_agent_mcp_tools()


In [ ]:
with write_demo_notebook("07_mcp_sample.ipynb") as path:
    _example_write_nb(
        str(path),
        "%%code\n"
        "#| default_exp sample\n"
        "def sample():\n"
        "    return 'ok'",
        replace=True,
    )
    text = capture_notebook_call(nb_overview, path, nb_path=str(path))
    assert "def sample():" in text
    print("captured notebook overview lines:", len(text.splitlines()))

Wrote 1 cells to nbs/data/07_mcp_sample.ipynb using replace and exported with nbdev
captured notebook overview lines: 2


In [ ]:
assert as_text(None) == ""
assert as_text({"ok": True}) == "{'ok': True}"
assert capture_call(lambda: "returned") == "returned"

In [ ]:
def _prints_and_returns():
    print("printed")
    return "returned"

In [ ]:
assert capture_call(_prints_and_returns) == "printed"

In [ ]:
def _prints_and_exits():
    print("before exit")
    raise SystemExit(7)

In [ ]:
try:
    capture_call(_prints_and_exits)
except RuntimeError as exc:
    assert "before exit" in str(exc)
    assert "SystemExit: 7" in str(exc)
else:
    raise AssertionError("SystemExit should be converted to RuntimeError for MCP tools")

In [ ]:
import time

In [ ]:
original_stdout = sys.stdout
outputs = []
errors = []
entered = threading.Event()

In [ ]:
def _slow_print(label, delay, signal=None):
    def inner():
        if signal is not None: signal.set()
        time.sleep(delay)
        print(label)
    return inner

In [ ]:
def _capture_worker(label, delay, signal=None):
    try:
        outputs.append(capture_call(_slow_print(label, delay, signal=signal)))
    except BaseException as exc:
        errors.append(exc)

In [ ]:
cell_stdout = sys.stdout
threads = [
    threading.Thread(target=_capture_worker, args=("first", 0.03, entered)),
    threading.Thread(target=_capture_worker, args=("second", 0.01)),
]
threads[0].start()
assert entered.wait(1)
threads[1].start()
for thread in threads: thread.join()

assert errors == []
assert sorted(outputs) == ["first", "second"]
assert sys.stdout is cell_stdout